# DLSS Neural Rendering for Colab — Unofficial CUDA Workflow

Experimental image/video workflow based on [MLX-DLSS](https://github.com/iamwavecut/MLX-DLSS), with CUDA/Triton optimizations. Not an NVIDIA product or native NGX invocation.

Tested on A100: 240 frames at 2416 × 1800, approximately 103 seconds processing including I/O, excluding calibration. Other GPUs are unverified. Not real-time.

Provide your own compatible model weights or DLL and review the applicable terms. No proprietary binaries or weights are distributed or downloaded by this notebook.

Run settings → setup → weights → benchmark → preview → export. Do not bypass failed numerical checks. PROCESSING_SCALE is internal supersampling, not output upscaling. See README.md for credits, limitations, and benchmark details.


In [ ]:
# Cell 1 — Settings. Set paths and review the license before running.
from pathlib import Path
from datetime import datetime

INPUT_PATH = Path('/content/input.mp4')  # PNG/JPEG image or FFmpeg-readable video
DLSSNR_DLL_PATH = Path('/content/nvngx_dlssnr.dll')  # Your own 310.8.0.0 DLL; ignored if WEIGHTS_PATH exists
WEIGHTS_PATH = Path('/content/portable_neural_rendering/weights/dlssnr-weights-logical.safetensors')
ACCEPT_MODEL_TERMS = False  # Set True only after checking your rights to use/extract these weights

DEVICE = 'auto'              # 'auto', 'cuda', 'cuda:0', or 'cpu'
ALLOW_CPU = False           # False prevents an unexpectedly slow CPU job
PRECISION = 'fast'          # CUDA float16; 'reference' is float32 and uses more memory
PROFILE = 'natural'         # standard / natural / cinematic / neutral
PROCESSING_SCALE = 1.0      # Internal work scale 1–4; output geometry stays unchanged
INTENSITY = 0.8             # 0 bypasses the effect; 1 is the upstream default
DETAIL_STRENGTH = 1.0      # 0–8; reduce if skin/hair looks over-sharpened
COLOUR_STRENGTH = 1.0      # 0–4
TEMPORAL = True            # Video history/optical-flow path; reset on scene changes
PREVIEW_FRAMES = 8         # Review before the full pass

WORK_DIR = Path('/content/portable_neural_rendering_fast')
REPO_DIR = WORK_DIR / 'MLX-DLSS'
OUTPUT_DIR = WORK_DIR / 'outputs'
UPSTREAM_COMMIT = '0ca2deab092fe6f3e331bf4f616271dbc64521d0'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
assert PROFILE in ('standard', 'natural', 'cinematic', 'neutral')
assert PRECISION in ('fast', 'reference')
assert 1 <= PROCESSING_SCALE <= 4 and 0 <= INTENSITY <= 1
assert 0 <= DETAIL_STRENGTH <= 8 and 0 <= COLOUR_STRENGTH <= 4
assert PREVIEW_FRAMES > 0
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Input:', INPUT_PATH, '| preview frames:', PREVIEW_FRAMES, '| device:', DEVICE)

# Speed target: enter the measured runtime for this SAME clip in the native notebook.
NATIVE_TARGET_SECONDS = 60.0    # target only; replace with a matched native measurement
MOTION_MAX_SIDE = 640          # motion guide only; rendering/output stay full resolution
GPU_RESIDENT_TEMPORAL = True   # v4; False restores the v3 temporal path for comparison
# PROCESSING_SCALE=2 does NOT double output dimensions. Keep the same value as
# the previous benchmark for an honest comparison; it is internal supersampling.
BENCHMARK_FRAMES = 4           # consecutive real frames, includes temporal behaviour
PROFILE_NETWORK = True        # one extra inference; writes operator timings + Chrome trace
CRF = 14
if PRECISION != 'fast' or ALLOW_CPU:
    raise ValueError('This fast CUDA experiment requires PRECISION="fast" and ALLOW_CPU=False')


In [ ]:
# Cell 2 — Preflight and install the pinned upstream PyTorch/video implementation.
import shutil, subprocess, sys
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f'Set INPUT_PATH to your image/video first: {INPUT_PATH}')
for program in ('git', 'ffmpeg', 'ffprobe'):
    if shutil.which(program) is None:
        raise RuntimeError(f'{program} is missing. Use a Colab runtime with FFmpeg installed.')
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    probe = torch.ones((32, 32), device='cuda')
    print('CUDA compute test:', float((probe @ probe).sum().item()))
if DEVICE.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError('CUDA requested but PyTorch cannot use this host driver/GPU.')
RESOLVED_DEVICE = ('cuda' if torch.cuda.is_available() else 'cpu') if DEVICE == 'auto' else DEVICE
if RESOLVED_DEVICE == 'cpu' and not ALLOW_CPU:
    raise RuntimeError('No usable CUDA GPU. Set ALLOW_CPU=True only for a tiny test; CPU video can be very slow.')
if RESOLVED_DEVICE == 'cpu' and PRECISION == 'fast':
    raise RuntimeError('CPU requires PRECISION="reference"; the fast path is CUDA/MPS float16.')
WORK_DIR.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/iamwavecut/MLX-DLSS.git', str(REPO_DIR)], check=True)
head = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if head != UPSTREAM_COMMIT:
    if subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Upstream checkout has local changes; use a clean Colab runtime.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', UPSTREAM_COMMIT], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', UPSTREAM_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip() == UPSTREAM_COMMIT
subprocess.run([sys.executable, '-m', 'pip', 'install', str(REPO_DIR / 'python') + '[video]'], check=True)
print('Pinned upstream ready:', UPSTREAM_COMMIT, '| device:', RESOLVED_DEVICE)

# Use the Triton matched to the installed PyTorch; do not blindly replace CUDA/Torch.
try:
    import triton
except ImportError as exc:
    raise RuntimeError('This runtime needs a CUDA PyTorch installation with matching Triton. '
                       'Use a standard Colab GPU runtime, then rerun setup.') from exc
print('Triton:', triton.__version__)
import base64
FAST_RUNTIME_DIR = WORK_DIR / 'fast_runtime'
FAST_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
(FAST_RUNTIME_DIR / 'fast_kernels.py').write_bytes(base64.b64decode('IiIiT3B0aW9uYWwgZnVzZWQgQ1VEQSBrZXJuZWxzIGZvciB0aGUgcGlubmVkIE1MWC1ETFNTIHJlY292ZXJlZCBncmFwaC4KCk5vIG1vZGVsIHdlaWdodHMgb3IgTlZJRElBIGJpbmFyaWVzIGFyZSBpbmNsdWRlZC4gRWFjaCBrZXJuZWwgbXVzdCBwYXNzIHRoZQp1cHN0cmVhbSBvcGVyYXRpb24gY29tcGFyaXNvbiBvbiB0aGUgYWN0dWFsIEdQVSBiZWZvcmUgaXQgaXMgZW5hYmxlZC4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBpbXBvcnRsaWIudXRpbAppbXBvcnQgc3lzCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRyaXRvbgppbXBvcnQgdHJpdG9uLmxhbmd1YWdlIGFzIHRsCgoKQHRyaXRvbi5qaXQKZGVmIF9lNCh4KToKICAgICMgU29mdHdhcmUgRTRNMyByb3VuZGluZyBhbHNvIHdvcmtzIG9uIEdQVXMgd2l0aG91dCBGUDggVGVuc29yIENvcmVzLgogICAgeCA9IHgudG8odGwuZmxvYXQzMikKICAgIG1hZyA9IHRsLm1pbmltdW0odGwuYWJzKHgpLCA0NDguMCkKICAgIG5vcm1hbCA9IHRsLm1heGltdW0obWFnLCAwLjAxNTYyNSkKICAgIGV4cG9uZW50ID0gKG5vcm1hbC50byh0bC5pbnQzMiwgYml0Y2FzdD1UcnVlKSA+PiAyMykgJiAyNTUKICAgIHN0ZXAgPSAoKGV4cG9uZW50IC0gMykgPDwgMjMpLnRvKHRsLmZsb2F0MzIsIGJpdGNhc3Q9VHJ1ZSkKICAgIHN0ZXAgPSB0bC53aGVyZShtYWcgPCAwLjAxNTYyNSwgMC4wMDE5NTMxMjUsIHN0ZXApCiAgICBzY2FsZWQgPSBtYWcgLyBzdGVwCiAgICBmbG9vciA9IHRsLmZsb29yKHNjYWxlZCkKICAgIGZyYWN0aW9uID0gc2NhbGVkIC0gZmxvb3IKICAgIG9kZCA9IChmbG9vci50byh0bC5pbnQzMikgJiAxKSAhPSAwCiAgICByb3VuZGVkID0gZmxvb3IgKyAoKGZyYWN0aW9uID4gMC41KSB8ICgoZnJhY3Rpb24gPT0gMC41KSAmIG9kZCkpLnRvKHRsLmZsb2F0MzIpCiAgICBvdXQgPSByb3VuZGVkICogc3RlcAogICAgcmV0dXJuIHRsLndoZXJlKHggPCAwLCAtb3V0LCBvdXQpCgoKQHRyaXRvbi5qaXQKZGVmIF9yb3VuZF9rZXJuZWwoWCwgWSwgTiwgQkxPQ0s6IHRsLmNvbnN0ZXhwcik6CiAgICBpID0gdGwucHJvZ3JhbV9pZCgwKSAqIEJMT0NLICsgdGwuYXJhbmdlKDAsIEJMT0NLKQogICAgeCA9IHRsLmxvYWQoWCArIGksIGkgPCBOLCBvdGhlcj0wKQogICAgdGwuc3RvcmUoWSArIGksIF9lNCh4KSwgaSA8IE4pCgoKZGVmIHJvdW5kX2U0KHZhbHVlKToKICAgIHggPSB2YWx1ZS5jb250aWd1b3VzKCkKICAgIG91dCA9IHRvcmNoLmVtcHR5X2xpa2UoeCkKICAgIF9yb3VuZF9rZXJuZWxbKHRyaXRvbi5jZGl2KHgubnVtZWwoKSwgMTAyNCksKV0oCiAgICAgICAgeCwgb3V0LCB4Lm51bWVsKCksIDEwMjQsIGVuYWJsZV9mcF9mdXNpb249RmFsc2UpCiAgICByZXR1cm4gb3V0CgoKQHRyaXRvbi5qaXQKZGVmIF9nYXRlX2tlcm5lbChYLCBZLCBOLCBCTE9DSzogdGwuY29uc3RleHByKToKICAgIGkgPSB0bC5wcm9ncmFtX2lkKDApICogQkxPQ0sgKyB0bC5hcmFuZ2UoMCwgQkxPQ0spCiAgICB4ID0gdGwubG9hZChYICsgaSwgaSA8IE4sIG90aGVyPTApLnRvKHRsLmZsb2F0MTYpCiAgICBjbGFtcGVkID0gdGwubWluaW11bSh0bC5tYXhpbXVtKHgudG8odGwuZmxvYXQzMiksIC00LjApLCA0LjApCiAgICBsaW5lYXIgPSAodGwuYWJzKGNsYW1wZWQpICogLTAuMDU1OTA4MjAzMTI1ICsgMC40NDcyNjU2MjUpLnRvKHRsLmZsb2F0MTYpCiAgICBnYXRlID0gKGNsYW1wZWQgKiBsaW5lYXIudG8odGwuZmxvYXQzMikgKyAwLjg5NDUzMTI1KS50byh0bC5mbG9hdDE2KQogICAgcmVzdWx0ID0gKHgudG8odGwuZmxvYXQzMikgKiBnYXRlLnRvKHRsLmZsb2F0MzIpKS50byh0bC5mbG9hdDE2KQogICAgdGwuc3RvcmUoWSArIGksIHJlc3VsdCwgaSA8IE4pCgoKZGVmIGdhdGVfYWN0aXZhdGlvbih2YWx1ZSk6CiAgICB4ID0gdmFsdWUuY29udGlndW91cygpCiAgICBvdXQgPSB0b3JjaC5lbXB0eV9saWtlKHgpCiAgICBfZ2F0ZV9rZXJuZWxbKHRyaXRvbi5jZGl2KHgubnVtZWwoKSwgMTAyNCksKV0oCiAgICAgICAgeCwgb3V0LCB4Lm51bWVsKCksIDEwMjQsIGVuYWJsZV9mcF9mdXNpb249RmFsc2UpCiAgICByZXR1cm4gb3V0CgoKQHRyaXRvbi5qaXQKZGVmIF9zb2Z0bWF4X2tlcm5lbChYLCBZLCBOOiB0bC5jb25zdGV4cHIsIEJMT0NLOiB0bC5jb25zdGV4cHIpOgogICAgcm93ID0gdGwucHJvZ3JhbV9pZCgwKQogICAgYyA9IHRsLmFyYW5nZSgwLCBCTE9DSykKICAgICMgRXZlcnkgbG9naWNhbCBwYWlyIHBhcnRpY2lwYXRlcyBpbiB0aGUgb3JpZ2luYWwgcGFja2VkIGhhbGYtYml0IGFmZmluZQogICAgIyB0cmFuc2Zvcm0sIGluY2x1ZGluZyB0aGUgY2FycnkgZnJvbSB0aGUgbG93IHRvIHRoZSBoaWdoIGhhbGZ3b3JkLgogICAgZXZlbiA9IGMgJiB+MQogICAgYSA9IHRsLmxvYWQoWCArIHJvdyAqIE4gKyBldmVuLCBldmVuIDwgTiwgb3RoZXI9MCkudG8odGwuZmxvYXQxNikudG8odGwuZmxvYXQzMikKICAgIGIgPSB0bC5sb2FkKFggKyByb3cgKiBOICsgZXZlbiArIDEsIGV2ZW4gKyAxIDwgTiwgb3RoZXI9MCkudG8odGwuZmxvYXQxNikudG8odGwuZmxvYXQzMikKICAgIGEgPSAoYSAqIDAuMDQ0OTIxODc1ICsgMS4zMDA3ODEyNSkudG8odGwuZmxvYXQxNikKICAgIGIgPSAoYiAqIDAuMDQ0OTIxODc1ICsgMS4zMDA3ODEyNSkudG8odGwuZmxvYXQxNikKICAgIGEgPSB0bC5taW5pbXVtKHRsLm1heGltdW0oYSwgMS4wMzEyNSksIDEuNTY5MzM1OTM3NSkudG8odGwuZmxvYXQxNikKICAgIGIgPSB0bC5taW5pbXVtKHRsLm1heGltdW0oYiwgMS4wMzEyNSksIDEuNTY5MzM1OTM3NSkudG8odGwuZmxvYXQxNikKICAgIHBhY2tlZCA9IGEudG8odGwudWludDE2LCBiaXRjYXN0PVRydWUpLnRvKHRsLnVpbnQzMikgfCAoYi50byh0bC51aW50MTYsIGJpdGNhc3Q9VHJ1ZSkudG8odGwudWludDMyKSA8PCAxNikKICAgIHRyYW5zZm9ybWVkID0gKHBhY2tlZCA8PCA1KSArIDB4N0ZGODgwMDAKICAgIGJpdHMgPSB0bC53aGVyZSgoYyAmIDEpID09IDAsIHRyYW5zZm9ybWVkICYgNjU1MzUsICh0cmFuc2Zvcm1lZCA+PiAxNikgJiA2NTUzNSkKICAgIHdlaWdodCA9IGJpdHMudG8odGwudWludDE2KS50byh0bC5mbG9hdDE2LCBiaXRjYXN0PVRydWUpCiAgICB0b3RhbCA9IHRsLnN1bSh0bC53aGVyZShjIDwgTiwgd2VpZ2h0LnRvKHRsLmZsb2F0MzIpLCAwKSwgMCkudG8odGwuZmxvYXQxNikKICAgIHJlY2lwcm9jYWwgPSB0bC5kaXZfcm4oMS4wLCB0b3RhbC50byh0bC5mbG9hdDMyKSkudG8odGwuZmxvYXQxNikKICAgIHByb2JhYmlsaXR5ID0gKHdlaWdodC50byh0bC5mbG9hdDMyKSAqIHJlY2lwcm9jYWwudG8odGwuZmxvYXQzMikpLnRvKHRsLmZsb2F0MTYpCiAgICB0bC5zdG9yZShZICsgcm93ICogTiArIGMsIF9lNChwcm9iYWJpbGl0eSksIGMgPCBOKQoKCmRlZiBzb2Z0bWF4KHZhbHVlKToKICAgIHggPSB2YWx1ZS5jb250aWd1b3VzKCkKICAgIG4gPSB4LnNoYXBlWy0xXQogICAgb3V0ID0gdG9yY2guZW1wdHlfbGlrZSh4KQogICAgX3NvZnRtYXhfa2VybmVsWyh4Lm51bWVsKCkgLy8gbiwpXSgKICAgICAgICB4LCBvdXQsIG4sIHRyaXRvbi5uZXh0X3Bvd2VyX29mXzIobiksIG51bV93YXJwcz00LAogICAgICAgIGVuYWJsZV9mcF9mdXNpb249RmFsc2UpCiAgICByZXR1cm4gb3V0CgoKQHRyaXRvbi5qaXQKZGVmIF9jb3NpbmVfa2VybmVsKFgsIFNDQUxFLCBZLCBST1dTOiB0bC5jb25zdGV4cHIsIEhFQURTOiB0bC5jb25zdGV4cHIsCiAgICAgICAgICAgICAgICAgICBUT0tFTlM6IHRsLmNvbnN0ZXhwciwgUzA6IHRsLmNvbnN0ZXhwciwgUzE6IHRsLmNvbnN0ZXhwciwKICAgICAgICAgICAgICAgICAgIFMyOiB0bC5jb25zdGV4cHIsIFMzOiB0bC5jb25zdGV4cHIsCiAgICAgICAgICAgICAgICAgICBIQVNfU0NBTEU6IHRsLmNvbnN0ZXhwciwgQkxPQ0tfUk9XUzogdGwuY29uc3RleHByKToKICAgIHIgPSB0bC5wcm9ncmFtX2lkKDApICogQkxPQ0tfUk9XUyArIHRsLmFyYW5nZSgwLCBCTE9DS19ST1dTKQogICAgYyA9IHRsLmFyYW5nZSgwLCA4KQogICAgaCA9IChyIC8vIFRPS0VOUykgJSBIRUFEUwogICAgYmFzZSA9IChyIC8vIChIRUFEUyAqIFRPS0VOUykpICogUzAgKyBoICogUzEgKyAociAlIFRPS0VOUykgKiBTMgogICAgb2Zmc2V0cyA9IGJhc2VbOiwgTm9uZV0gKyBjW05vbmUsIDpdICogUzMKICAgIG1hc2sgPSByWzosIE5vbmVdIDwgUk9XUwogICAgYSA9IHRsLmxvYWQoWCArIG9mZnNldHMsIG1hc2ssIG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICBiID0gdGwubG9hZChYICsgb2Zmc2V0cyArIDggKiBTMywgbWFzaywgb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgIGQgPSB0bC5sb2FkKFggKyBvZmZzZXRzICsgMTYgKiBTMywgbWFzaywgb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgIGUgPSB0bC5sb2FkKFggKyBvZmZzZXRzICsgMjQgKiBTMywgbWFzaywgb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgIGZpcnN0ID0gKGIgKiBiICsgKGEgKiBhKS50byh0bC5mbG9hdDE2KS50byh0bC5mbG9hdDMyKSkudG8odGwuZmxvYXQxNikKICAgIHNlY29uZCA9IChlICogZSArIChkICogZCkudG8odGwuZmxvYXQxNikudG8odGwuZmxvYXQzMikpLnRvKHRsLmZsb2F0MTYpCiAgICBwYXJ0aWFsID0gKGZpcnN0LnRvKHRsLmZsb2F0MzIpICsgc2Vjb25kLnRvKHRsLmZsb2F0MzIpKS50byh0bC5mbG9hdDE2KQogICAgaWR4NCA9IHRsLmJyb2FkY2FzdF90bygoYyBeIDQpW05vbmUsIDpdLCAoQkxPQ0tfUk9XUywgOCkpCiAgICB4b3IyID0gKHBhcnRpYWwudG8odGwuZmxvYXQzMikgKyB0bC5nYXRoZXIocGFydGlhbCwgaWR4NCwgMSkudG8odGwuZmxvYXQzMikpLnRvKHRsLmZsb2F0MTYpCiAgICBpZHgyID0gdGwuYnJvYWRjYXN0X3RvKChjIF4gMilbTm9uZSwgOl0sIChCTE9DS19ST1dTLCA4KSkKICAgIHhvcjEgPSAoeG9yMi50byh0bC5mbG9hdDMyKSArIHRsLmdhdGhlcih4b3IyLCBpZHgyLCAxKS50byh0bC5mbG9hdDMyKSkudG8odGwuZmxvYXQxNikKICAgIG4wID0gdGwuc3VtKHRsLndoZXJlKGNbTm9uZSwgOl0gPT0gMCwgeG9yMS50byh0bC5mbG9hdDMyKSwgMCksIDEpCiAgICBuMSA9IHRsLnN1bSh0bC53aGVyZShjW05vbmUsIDpdID09IDEsIHhvcjEudG8odGwuZmxvYXQzMiksIDApLCAxKQogICAgbm9ybSA9IChuMCArIG4xKS50byh0bC5mbG9hdDE2KS50byh0bC5mbG9hdDMyKQogICAgcmVjaXByb2NhbCA9IHRsLnJzcXJ0KHRsLm1heGltdW0obm9ybSwgMC4wMDAwNjE5ODg4MzA1NjY0MDYyNSkpLnRvKHRsLmZsb2F0MTYpLnRvKHRsLmZsb2F0MzIpCiAgICBuYSA9IChhICogcmVjaXByb2NhbFs6LCBOb25lXSkudG8odGwuZmxvYXQxNikKICAgIG5iID0gKGIgKiByZWNpcHJvY2FsWzosIE5vbmVdKS50byh0bC5mbG9hdDE2KQogICAgbmQgPSAoZCAqIHJlY2lwcm9jYWxbOiwgTm9uZV0pLnRvKHRsLmZsb2F0MTYpCiAgICBuZSA9IChlICogcmVjaXByb2NhbFs6LCBOb25lXSkudG8odGwuZmxvYXQxNikKICAgIGlmIEhBU19TQ0FMRToKICAgICAgICBzY2FsZSA9IHRsLmxvYWQoU0NBTEUgKyBoLCByIDwgUk9XUywgb3RoZXI9MSkudG8odGwuZmxvYXQxNikudG8odGwuZmxvYXQzMikKICAgICAgICBuYSA9IChuYS50byh0bC5mbG9hdDMyKSAqIHNjYWxlWzosIE5vbmVdKS50byh0bC5mbG9hdDE2KQogICAgICAgIG5iID0gKG5iLnRvKHRsLmZsb2F0MzIpICogc2NhbGVbOiwgTm9uZV0pLnRvKHRsLmZsb2F0MTYpCiAgICAgICAgbmQgPSAobmQudG8odGwuZmxvYXQzMikgKiBzY2FsZVs6LCBOb25lXSkudG8odGwuZmxvYXQxNikKICAgICAgICBuZSA9IChuZS50byh0bC5mbG9hdDMyKSAqIHNjYWxlWzosIE5vbmVdKS50byh0bC5mbG9hdDE2KQogICAgZGVzdCA9IHJbOiwgTm9uZV0gKiAzMiArIGNbTm9uZSwgOl0KICAgIHRsLnN0b3JlKFkgKyBkZXN0LCBfZTQobmEpLCBtYXNrKQogICAgdGwuc3RvcmUoWSArIGRlc3QgKyA4LCBfZTQobmIpLCBtYXNrKQogICAgdGwuc3RvcmUoWSArIGRlc3QgKyAxNiwgX2U0KG5kKSwgbWFzaykKICAgIHRsLnN0b3JlKFkgKyBkZXN0ICsgMjQsIF9lNChuZSksIG1hc2spCgoKZGVmIGNvc2luZV9wdWJsaXNoKHZhbHVlLCBzY2FsZT1Ob25lKToKICAgIG91dCA9IHRvcmNoLmVtcHR5KHZhbHVlLnNoYXBlLCBkZXZpY2U9dmFsdWUuZGV2aWNlLCBkdHlwZT12YWx1ZS5kdHlwZSkKICAgIHJvd3MgPSB2YWx1ZS5udW1lbCgpIC8vIDMyCiAgICBfY29zaW5lX2tlcm5lbFsodHJpdG9uLmNkaXYocm93cywgOCksKV0oCiAgICAgICAgdmFsdWUsIHZhbHVlIGlmIHNjYWxlIGlzIE5vbmUgZWxzZSBzY2FsZSwgb3V0LAogICAgICAgIHJvd3MsIHZhbHVlLnNoYXBlWzFdLCB2YWx1ZS5zaGFwZVsyXSwgKnZhbHVlLnN0cmlkZSgpLAogICAgICAgIHNjYWxlIGlzIG5vdCBOb25lLCA4LCBudW1fd2FycHM9NCwgZW5hYmxlX2ZwX2Z1c2lvbj1GYWxzZSkKICAgIHJldHVybiBvdXQKCgpkZWYgdmFsaWRhdGVfa2VybmVscyhkZXZpY2UpOgogICAgIiIiRW5hYmxlIG9ubHkga2VybmVscyB3aG9zZSB0ZXN0IG91dHB1dHMgZXhhY3RseSBlcXVhbCB0aGUgcGlubmVkIHJlZmVyZW5jZS4KCiAgICBBIGZhaWxlZCBrZXJuZWwgaXMgcmVwb3J0ZWQgYW5kIGxlZnQgb24gdGhlIHJlZmVyZW5jZSBpbXBsZW1lbnRhdGlvbi4KICAgIEEgc2VwYXJhdGUgcmVhbC1mcmFtZSB0ZW1wb3JhbCBjb21wYXJpc29uIGlzIHJlcXVpcmVkIGJ5IHRoZSBub3RlYm9vay4KICAgICIiIgogICAgZnJvbSBtbHhkbHNzIGltcG9ydCBtb2RlbCBhcyByZWZlcmVuY2UKICAgIGdlbiA9IHRvcmNoLkdlbmVyYXRvcihkZXZpY2U9ImNwdSIpLm1hbnVhbF9zZWVkKDgxNikKICAgIHJlcG9ydCA9IHt9CgogICAgZGVmIGNoZWNrKG5hbWUsIGNhc2VzKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZvciBleHBlY3RlZCwgYWN0dWFsIGluIGNhc2VzKCk6CiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKGRldmljZSkKICAgICAgICAgICAgICAgIGlmIG5vdCB0b3JjaC5lcXVhbChleHBlY3RlZCwgYWN0dWFsKToKICAgICAgICAgICAgICAgICAgICBkaWZmID0gKGV4cGVjdGVkLmZsb2F0KCkgLSBhY3R1YWwuZmxvYXQoKSkuYWJzKCkKICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJub3QgZXhhY3Q6IG1heD17ZGlmZi5tYXgoKS5pdGVtKCk6Z30sIG1lYW49e2RpZmYubWVhbigpLml0ZW0oKTpnfSIpCiAgICAgICAgICAgIHJlcG9ydFtuYW1lXSA9IHsiZW5hYmxlZCI6IFRydWUsICJyZXN1bHQiOiAiZXhhY3Qgb24gdGVzdCBpbnB1dHMifQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICByZXBvcnRbbmFtZV0gPSB7ImVuYWJsZWQiOiBGYWxzZSwgInJlc3VsdCI6IHN0cihleGMpfQogICAgICAgIHByaW50KGYiS2VybmVsIHtuYW1lfToge3JlcG9ydFtuYW1lXX0iLCBmbHVzaD1UcnVlKQoKICAgIGRlZiByb3VuZF9jYXNlcygpOgogICAgICAgICMgRXZlcnkgZmluaXRlIGZsb2F0MTYgZW5jb2RpbmcsIGluY2x1ZGluZyBhbGwgaGFsZi13YXkgYm91bmRhcmllcy4KICAgICAgICBhbGxfaGFsZiA9IHRvcmNoLmFyYW5nZSg2NTUzNiwgZHR5cGU9dG9yY2guaW50MzIpLnRvKHRvcmNoLmludDE2KS52aWV3KHRvcmNoLmZsb2F0MTYpCiAgICAgICAgeCA9IGFsbF9oYWxmW3RvcmNoLmlzZmluaXRlKGFsbF9oYWxmKV0udG8oZGV2aWNlKQogICAgICAgIHlpZWxkIHJlZmVyZW5jZS5lNG0zX3JvdW5kX3RyaXAoeCksIHJvdW5kX2U0KHgpCgogICAgZGVmIHNvZnRtYXhfY2FzZXMoKToKICAgICAgICBmb3IgbiBpbiAoNjQsIDk2LCAyNTYsIDUxMik6CiAgICAgICAgICAgIHggPSAodG9yY2gucmFuZG4oKDI1NywgbiksIGdlbmVyYXRvcj1nZW4pICogMTIpLmhhbGYoKS50byhkZXZpY2UpCiAgICAgICAgICAgIHhbMF0gPSAwCiAgICAgICAgICAgIHlpZWxkIHJlZmVyZW5jZS52ZW5kb3JfYXBwcm94aW1hdGVfc29mdG1heCh4KSwgc29mdG1heCh4KQoKICAgIGRlZiBnYXRlX2Nhc2VzKCk6CiAgICAgICAgYWxsX2hhbGYgPSB0b3JjaC5hcmFuZ2UoNjU1MzYsIGR0eXBlPXRvcmNoLmludDMyKS50byh0b3JjaC5pbnQxNikudmlldyh0b3JjaC5mbG9hdDE2KQogICAgICAgIHggPSBhbGxfaGFsZlt0b3JjaC5pc2Zpbml0ZShhbGxfaGFsZildLnRvKGRldmljZSkKICAgICAgICB5aWVsZCByZWZlcmVuY2UucXVhZHJhdGljX2dhdGVfYWN0aXZhdGlvbih4KSwgZ2F0ZV9hY3RpdmF0aW9uKHgpCgogICAgZGVmIGNvc2luZV9jYXNlcygpOgogICAgICAgIGZvciBoZWFkcyBpbiAoMSwgMiwgMTYsIDMyKToKICAgICAgICAgICAgZm9yIGFtcGxpdHVkZSBpbiAoMC4wMDAxLCAwLjEsIDEuMCwgNC4wKToKICAgICAgICAgICAgICAgIHggPSAodG9yY2gucmFuZG4oKDIsIDY1LCBoZWFkcywgMzIpLCBnZW5lcmF0b3I9Z2VuKSAqIGFtcGxpdHVkZSkuaGFsZigpLnRvKGRldmljZSkucGVybXV0ZSgwLCAyLCAxLCAzKQogICAgICAgICAgICAgICAgeFswLCA6LCAwXSA9IDAKICAgICAgICAgICAgICAgIGZvciBzY2FsZSBpbiAoTm9uZSwgdG9yY2gubGluc3BhY2UoMC41LCA4LCBoZWFkcywgZGV2aWNlPWRldmljZSkuaGFsZigpKToKICAgICAgICAgICAgICAgICAgICB5aWVsZCByZWZlcmVuY2UudmVuZG9yX2Nvc2luZV9wdWJsaXNoKHgsIHNjYWxlKSwgY29zaW5lX3B1Ymxpc2goeCwgc2NhbGUpCgogICAgd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpLCB0b3JjaC5jdWRhLmRldmljZShkZXZpY2UpOgogICAgICAgIGNoZWNrKCJyb3VuZF9lNCIsIHJvdW5kX2Nhc2VzKQogICAgICAgIGNoZWNrKCJnYXRlX2FjdGl2YXRpb24iLCBnYXRlX2Nhc2VzKQogICAgICAgIGNoZWNrKCJzb2Z0bWF4Iiwgc29mdG1heF9jYXNlcykKICAgICAgICBjaGVjaygiY29zaW5lX3B1Ymxpc2giLCBjb3NpbmVfY2FzZXMpCiAgICBpZiBub3QgYW55KGVudHJ5WyJlbmFibGVkIl0gZm9yIGVudHJ5IGluIHJlcG9ydC52YWx1ZXMoKSk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTm8gZnVzZWQga2VybmVsIHBhc3NlZCB2YWxpZGF0aW9uIG9uIHRoaXMgaG9zdDoge3JlcG9ydH0iKQogICAgcmV0dXJuIHJlcG9ydAoKCmRlZiBidWlsZF9mYXN0X21vZGVsKHdlaWdodHMsIGRldmljZSwgdmFsaWRhdGlvbik6CiAgICAiIiJQcml2YXRlIGNvcHkgb2YgdGhlIHBpbm5lZCBncmFwaDsgdXBzdHJlYW0gbW9kdWxlIHN0YXlzIGF2YWlsYWJsZSBmb3IgQS9CLiIiIgogICAgZnJvbSBtbHhkbHNzIGltcG9ydCBtb2RlbCBhcyByZWZlcmVuY2UKICAgIG5hbWUgPSAiX3BvcnRhYmxlX25yX2Zhc3RfcmVmZXJlbmNlIgogICAgc3BlYyA9IGltcG9ydGxpYi51dGlsLnNwZWNfZnJvbV9maWxlX2xvY2F0aW9uKG5hbWUsIHJlZmVyZW5jZS5fX2ZpbGVfXykKICAgIG1vZHVsZSA9IGltcG9ydGxpYi51dGlsLm1vZHVsZV9mcm9tX3NwZWMoc3BlYykKICAgIHN5cy5tb2R1bGVzW25hbWVdID0gbW9kdWxlCiAgICBzcGVjLmxvYWRlci5leGVjX21vZHVsZShtb2R1bGUpCiAgICAjIEJhdGNoIGluZGVwZW5kZW50IG91dHB1dC1oZWFkL2JyYW5jaCBHRU1NcywgYnV0IHByZXNlcnZlIHRoZSBvcmlnaW5hbAogICAgIyBpbnB1dC1oZWFkIGFuZCBicmFuY2ggYWNjdW11bGF0aW9uIG9yZGVyIChpbmNsdWRpbmcgaGFsZiByb3VuZGluZykuCiAgICAjIE5vIGNvbmNhdGVuYXRlZCBsYXJnZSBHRU1NOiB0aGF0IHdvdWxkIGNoYW5nZSB0aGUgcmVkdWN0aW9uIHNlbWFudGljcy4KICAgIGRlZiBiYXRjaGVkX2JyYW5jaGVkKHZhbHVlLCAqLCBleHBhbnNpb25fd2VpZ2h0LCBicmFuY2hfcHJvamVjdGlvbl93ZWlnaHQsCiAgICAgICAgICAgICAgICAgICAgICAgICBvdXRwdXRfcHJvamVjdGlvbl93ZWlnaHQpOgogICAgICAgIGdyb3VwcyA9IHZhbHVlLnNoYXBlWy0xXSAvLyAzMgogICAgICAgIGxlYWQgPSB2YWx1ZS5zaGFwZVs6LTFdCiAgICAgICAgaW5wdXRzID0gdmFsdWUucmVzaGFwZSgtMSwgZ3JvdXBzLCAzMikKICAgICAgICBleHBhbmRlZCA9IDAKICAgICAgICBmb3IgaW5kZXggaW4gcmFuZ2UoZ3JvdXBzKToKICAgICAgICAgICAgZXhwYW5kZWQgPSBleHBhbmRlZCArIHRvcmNoLm1hdG11bChpbnB1dHNbOiwgaW5kZXgsIDpdLCBleHBhbnNpb25fd2VpZ2h0WzosIDosIGluZGV4XSkKICAgICAgICBhY3RpdmF0ZWQgPSBtb2R1bGUuZTRtM19yb3VuZF90cmlwKG1vZHVsZS5xdWFkcmF0aWNfZ2F0ZV9hY3RpdmF0aW9uKGV4cGFuZGVkKSkKICAgICAgICBicmFuY2hlcyA9IHRvcmNoLm1hdG11bChhY3RpdmF0ZWQsIGJyYW5jaF9wcm9qZWN0aW9uX3dlaWdodCkKICAgICAgICBzdW1tZWQgPSBicmFuY2hlc1s6LCAwXSArIGJyYW5jaGVzWzosIDFdCiAgICAgICAgc3VtbWVkID0gc3VtbWVkICsgYnJhbmNoZXNbOiwgMl0KICAgICAgICBzdW1tZWQgPSBzdW1tZWQgKyBicmFuY2hlc1s6LCAzXQogICAgICAgIGhlYWRzID0gbW9kdWxlLmU0bTNfcm91bmRfdHJpcChzdW1tZWQpCiAgICAgICAgbWVyZ2VkID0gaGVhZHMucGVybXV0ZSgxLCAwLCAyKS5yZXNoYXBlKCpsZWFkLCBncm91cHMgKiAzMikKICAgICAgICByZXR1cm4gbWVyZ2VkIEAgb3V0cHV0X3Byb2plY3Rpb25fd2VpZ2h0CgogICAgbW9kdWxlLmJyYW5jaGVkX2ZlZWRfZm9yd2FyZCA9IGJhdGNoZWRfYnJhbmNoZWQKICAgIGRlZiBiYXRjaGVkX3NwbGl0KHZhbHVlLCAqLCBmaXJzdF9wcm9qZWN0aW9uX3dlaWdodCwgZXhwYW5kX3dlaWdodCwgcHJvamVjdF93ZWlnaHQpOgogICAgICAgIGhpZGRlbiA9IG1vZHVsZS5lNG0zX3JvdW5kX3RyaXAodmFsdWUgQCBmaXJzdF9wcm9qZWN0aW9uX3dlaWdodCkKICAgICAgICBncm91cHMgPSBoaWRkZW4uc2hhcGVbLTFdIC8vIDY0CiAgICAgICAgZ3JvdXBlZCA9IGhpZGRlbi5yZXNoYXBlKC0xLCBncm91cHMsIDY0KS5wZXJtdXRlKDEsIDAsIDIpCiAgICAgICAgZXhwYW5kZWQgPSB0b3JjaC5ibW0oZ3JvdXBlZCwgZXhwYW5kX3dlaWdodCkKICAgICAgICBwcm9qZWN0ZWQgPSB0b3JjaC5ibW0obW9kdWxlLnF1YWRyYXRpY19nYXRlX2FjdGl2YXRpb24oZXhwYW5kZWQpLCBwcm9qZWN0X3dlaWdodCkKICAgICAgICBtZXJnZWQgPSBwcm9qZWN0ZWQucGVybXV0ZSgxLCAwLCAyKS5yZXNoYXBlKCpoaWRkZW4uc2hhcGUpCiAgICAgICAgcmV0dXJuIG1vZHVsZS5lNG0zX3JvdW5kX3RyaXAobWVyZ2VkKQoKICAgIG1vZHVsZS5zcGxpdF9ncm91cF9mZWVkX2ZvcndhcmQgPSBiYXRjaGVkX3NwbGl0CiAgICBpZiB2YWxpZGF0aW9uWyJyb3VuZF9lNCJdWyJlbmFibGVkIl06CiAgICAgICAgb3JpZ2luYWxfcm91bmQgPSBtb2R1bGUuZTRtM19yb3VuZF90cmlwCiAgICAgICAgbW9kdWxlLmU0bTNfcm91bmRfdHJpcCA9IGxhbWJkYSB4OiByb3VuZF9lNCh4KSBpZiB4LmlzX2N1ZGEgYW5kIHguZHR5cGUgPT0gdG9yY2guZmxvYXQxNiBlbHNlIG9yaWdpbmFsX3JvdW5kKHgpCiAgICBpZiB2YWxpZGF0aW9uWyJnYXRlX2FjdGl2YXRpb24iXVsiZW5hYmxlZCJdOgogICAgICAgIG9yaWdpbmFsX2dhdGUgPSBtb2R1bGUucXVhZHJhdGljX2dhdGVfYWN0aXZhdGlvbgogICAgICAgIG1vZHVsZS5xdWFkcmF0aWNfZ2F0ZV9hY3RpdmF0aW9uID0gbGFtYmRhIHg6IGdhdGVfYWN0aXZhdGlvbih4KSBpZiB4LmlzX2N1ZGEgYW5kIHguZHR5cGUgPT0gdG9yY2guZmxvYXQxNiBlbHNlIG9yaWdpbmFsX2dhdGUoeCkKICAgIGlmIHZhbGlkYXRpb25bInNvZnRtYXgiXVsiZW5hYmxlZCJdOgogICAgICAgIG9yaWdpbmFsX3NvZnRtYXggPSBtb2R1bGUudmVuZG9yX2FwcHJveGltYXRlX3NvZnRtYXgKICAgICAgICBtb2R1bGUudmVuZG9yX2FwcHJveGltYXRlX3NvZnRtYXggPSBsYW1iZGEgeDogc29mdG1heCh4KSBpZiB4LmlzX2N1ZGEgYW5kIHguZHR5cGUgPT0gdG9yY2guZmxvYXQxNiBhbmQgeC5zaGFwZVstMV0gJSAyID09IDAgYW5kIHguc2hhcGVbLTFdIDw9IDQwOTYgZWxzZSBvcmlnaW5hbF9zb2Z0bWF4KHgpCiAgICBpZiB2YWxpZGF0aW9uWyJjb3NpbmVfcHVibGlzaCJdWyJlbmFibGVkIl06CiAgICAgICAgb3JpZ2luYWxfY29zaW5lID0gbW9kdWxlLnZlbmRvcl9jb3NpbmVfcHVibGlzaAogICAgICAgIG1vZHVsZS52ZW5kb3JfY29zaW5lX3B1Ymxpc2ggPSBsYW1iZGEgeCwgc2NhbGU9Tm9uZTogY29zaW5lX3B1Ymxpc2goeCwgc2NhbGUpIGlmIHguaXNfY3VkYSBhbmQgeC5kdHlwZSA9PSB0b3JjaC5mbG9hdDE2IGFuZCB4Lm5kaW0gPT0gNCBhbmQgeC5zaGFwZVstMV0gPT0gMzIgZWxzZSBvcmlnaW5hbF9jb3NpbmUoeCwgc2NhbGUpCiAgICBtb2RlbCA9IG1vZHVsZS5OZXVyYWxSZW5kZXJpbmdNb2RlbCh3ZWlnaHRzKS5oYWxmKCkudG8oZGV2aWNlKS5ldmFsKCkKICAgICMgQmlhcyBsYXlvdXQgaXMgY29uc3RhbnQgYWNyb3NzIGV2ZXJ5IGZyYW1lLiBSZXNvbHZlIGl0IG9uY2UsIG9uLWRldmljZS4KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGZvciBrZXkgaW4gbW9kZWwuX3dlaWdodF9hdHRyaWJ1dGVzOgogICAgICAgICAgICBpZiBrZXkuZW5kc3dpdGgoIi5hdHRuX2JpYXMiKToKICAgICAgICAgICAgICAgIGJpYXMgPSBtb2RlbC53ZWlnaHQoa2V5KQogICAgICAgICAgICAgICAgaWYgYmlhcy5zaGFwZVswXSBpbiAoMSwgMTYpOgogICAgICAgICAgICAgICAgICAgIGJpYXMuY29weV8ocmVmZXJlbmNlLnJlY292ZXJfYXR0ZW50aW9uX2JpYXNfbGF5b3V0KGJpYXMpKQogICAgbW9kdWxlLnVzZXNfZnJhZ21lbnRfc3dpenpsZSA9IGxhbWJkYSBpbmRleCwgaGVhZHM6IEZhbHNlCiAgICByZXR1cm4gbW9kZWwK'))
(FAST_RUNTIME_DIR / 'graph_replay.py').write_bytes(base64.b64decode('IiIiRml4ZWQtc2hhcGUgQ1VEQSByZXBsYXkgd2l0aCBjaGFuZ2VkLWlucHV0IGNoZWNrcyBhbmQgbWVhc3VyZWQgc2VsZWN0aW9uLiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdGltZQppbXBvcnQgdG9yY2gKCgpjbGFzcyBSZXBsYXlNb2RlbDoKICAgICIiIlNpbmdsZS1zdHJlYW0sIHNpbmdsZS1zaGFwZSBjYWNoZS4gUmV0dXJuZWQgc3RvcmFnZSBpcyByZXVzZWQgbmV4dCBjYWxsLgoKICAgIFRoZSBjYWxsZXIgY29uc3VtZXMvY29tcG9zZXMgb3V0cHV0IGJlZm9yZSB0aGUgbmV4dCBjYWxsLiBUZW1wb3JhbCBzdGF0ZSBpcwogICAgaW4gdGhlIGlucHV0IHRlbnNvciwgbm90IGluIHRoaXMgZ3JhcGgsIGFuZCBpcyBjb3BpZWQgb24gRVZFUlkgcmVwbGF5LgogICAgIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kZWwpOgogICAgICAgIHNlbGYubW9kZWwgPSBtb2RlbAogICAgICAgIHNlbGYuZ3JhcGggPSBzZWxmLmlucHV0ID0gc2VsZi5vdXRwdXQgPSBOb25lCiAgICAgICAgc2VsZi5rZXkgPSBOb25lCiAgICAgICAgc2VsZi5yZXBvcnRzID0gW10KCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgc2lnbmF0dXJlKHgpOgogICAgICAgIHJldHVybiAodHVwbGUoeC5zaGFwZSksIHguZHR5cGUsIHguZGV2aWNlKQoKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBtZWFzdXJlZChmbiwgZGV2aWNlLCByZXBlYXRzPTMpOgogICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoZGV2aWNlKQogICAgICAgIHN0YXJ0ZWQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgZm9yIF8gaW4gcmFuZ2UocmVwZWF0cyk6CiAgICAgICAgICAgIGZuKCkKICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKGRldmljZSkKICAgICAgICByZXR1cm4gKHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydGVkKSAvIHJlcGVhdHMKCiAgICBAdG9yY2guaW5mZXJlbmNlX21vZGUoKQogICAgZGVmIHByZXBhcmUoc2VsZiwgeCk6CiAgICAgICAgc2VsZi5rZXkgPSBzZWxmLnNpZ25hdHVyZSh4KQogICAgICAgIHNlbGYuZ3JhcGggPSBzZWxmLmlucHV0ID0gc2VsZi5vdXRwdXQgPSBOb25lCiAgICAgICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICByZXBvcnQgPSB7InNoYXBlIjogbGlzdCh4LnNoYXBlKSwgImVuYWJsZWQiOiBGYWxzZX0KICAgICAgICBzZWxmLnJlcG9ydHMuYXBwZW5kKHJlcG9ydCkKICAgICAgICAjIEF2b2lkIGhpZGluZyBhIGZhaWxlZCBjYXB0dXJlIGJlaGluZCBlbmRsZXNzIGF0dGVtcHRzIGV2ZXJ5IGZyYW1lLgogICAgICAgIHRyeToKICAgICAgICAgICAgd2l0aCB0b3JjaC5jdWRhLmRldmljZSh4LmRldmljZSk6CiAgICAgICAgICAgICAgICBzZWxmLmlucHV0ID0geC5jbG9uZSgpCiAgICAgICAgICAgICAgICBzdHJlYW0gPSB0b3JjaC5jdWRhLlN0cmVhbShkZXZpY2U9eC5kZXZpY2UpCiAgICAgICAgICAgICAgICBzdHJlYW0ud2FpdF9zdHJlYW0odG9yY2guY3VkYS5jdXJyZW50X3N0cmVhbSh4LmRldmljZSkpCiAgICAgICAgICAgICAgICB3aXRoIHRvcmNoLmN1ZGEuc3RyZWFtKHN0cmVhbSk6CiAgICAgICAgICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2UoMyk6CiAgICAgICAgICAgICAgICAgICAgICAgIHNlbGYubW9kZWwoc2VsZi5pbnB1dCkKICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEuY3VycmVudF9zdHJlYW0oeC5kZXZpY2UpLndhaXRfc3RyZWFtKHN0cmVhbSkKICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoeC5kZXZpY2UpCiAgICAgICAgICAgICAgICBncmFwaCA9IHRvcmNoLmN1ZGEuQ1VEQUdyYXBoKCkKICAgICAgICAgICAgICAgIHdpdGggdG9yY2guY3VkYS5ncmFwaChncmFwaCwgc3RyZWFtPXN0cmVhbSk6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5vdXRwdXQgPSBzZWxmLm1vZGVsKHNlbGYuaW5wdXQpCiAgICAgICAgICAgICAgICAjIFR3byBkaXN0aW5jdCBmZWF0dXJlIHRlbnNvcnMgZGV0ZWN0IGFjY2lkZW50YWxseSBmcm96ZW4gaW5wdXQuCiAgICAgICAgICAgICAgICBmb3IgcHJvYmUgaW4gKHgsIHggKiAwLjg3NSk6CiAgICAgICAgICAgICAgICAgICAgZXhwZWN0ZWQgPSBzZWxmLm1vZGVsKHByb2JlKS5jbG9uZSgpCiAgICAgICAgICAgICAgICAgICAgc2VsZi5pbnB1dC5jb3B5Xyhwcm9iZSkKICAgICAgICAgICAgICAgICAgICBncmFwaC5yZXBsYXkoKQogICAgICAgICAgICAgICAgICAgIGlmIG5vdCB0b3JjaC5pc2Zpbml0ZShzZWxmLm91dHB1dCkuYWxsKCkuaXRlbSgpIG9yIG5vdCB0b3JjaC5lcXVhbChleHBlY3RlZCwgc2VsZi5vdXRwdXQpOgogICAgICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkdyYXBoIHJlcGxheSBjaGFuZ2VkIG1vZGVsIG91dHB1dDsgcmV0YWluaW5nIGVhZ2VyIG1vZGVsIikKICAgICAgICAgICAgICAgIHNlbGYuaW5wdXQuY29weV8oeCkKICAgICAgICAgICAgICAgIGVhZ2VyX3NlY29uZHMgPSBzZWxmLm1lYXN1cmVkKGxhbWJkYTogc2VsZi5tb2RlbCh4KSwgeC5kZXZpY2UpCiAgICAgICAgICAgICAgICBkZWYgcmVwbGF5KCk6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5pbnB1dC5jb3B5Xyh4KQogICAgICAgICAgICAgICAgICAgIGdyYXBoLnJlcGxheSgpCiAgICAgICAgICAgICAgICByZXBsYXlfc2Vjb25kcyA9IHNlbGYubWVhc3VyZWQocmVwbGF5LCB4LmRldmljZSkKICAgICAgICAgICAgICAgIHJlcG9ydC51cGRhdGUoZWFnZXJfc2Vjb25kcz1lYWdlcl9zZWNvbmRzLCByZXBsYXlfc2Vjb25kcz1yZXBsYXlfc2Vjb25kcywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3BlZWR1cD1lYWdlcl9zZWNvbmRzIC8gcmVwbGF5X3NlY29uZHMsIGNoYW5nZWRfaW5wdXRfZXhhY3Q9VHJ1ZSkKICAgICAgICAgICAgICAgIGlmIHJlcGxheV9zZWNvbmRzIDwgZWFnZXJfc2Vjb25kczoKICAgICAgICAgICAgICAgICAgICBzZWxmLmdyYXBoID0gZ3JhcGgKICAgICAgICAgICAgICAgICAgICByZXBvcnRbImVuYWJsZWQiXSA9IFRydWUKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgcmVwb3J0WyJyZWFzb24iXSA9ICJSZXBsYXkgd2FzIG5vdCBmYXN0ZXIgb24gdGhpcyBzaGFwZSIKICAgICAgICBleGNlcHQgKFJ1bnRpbWVFcnJvciwgdG9yY2guY3VkYS5PdXRPZk1lbW9yeUVycm9yKSBhcyBleGM6CiAgICAgICAgICAgIHJlcG9ydFsicmVhc29uIl0gPSBzdHIoZXhjKQogICAgICAgIGlmIHNlbGYuZ3JhcGggaXMgTm9uZToKICAgICAgICAgICAgc2VsZi5pbnB1dCA9IHNlbGYub3V0cHV0ID0gTm9uZQogICAgICAgIHJlcG9ydFsic2V0dXBfc2Vjb25kcyJdID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQKICAgICAgICBwcmludCgiQ1VEQSBncmFwaDoiLCByZXBvcnQsIGZsdXNoPVRydWUpCgogICAgQHRvcmNoLmluZmVyZW5jZV9tb2RlKCkKICAgIGRlZiBfX2NhbGxfXyhzZWxmLCB4KToKICAgICAgICBpZiBzZWxmLnNpZ25hdHVyZSh4KSAhPSBzZWxmLmtleToKICAgICAgICAgICAgc2VsZi5wcmVwYXJlKHgpCiAgICAgICAgaWYgc2VsZi5ncmFwaCBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gc2VsZi5tb2RlbCh4KQogICAgICAgIHNlbGYuaW5wdXQuY29weV8oeCkKICAgICAgICBzZWxmLmdyYXBoLnJlcGxheSgpCiAgICAgICAgcmV0dXJuIHNlbGYub3V0cHV0CgoKQHRvcmNoLmluZmVyZW5jZV9tb2RlKCkKZGVmIHByb2ZpbGVfbmV0d29yayhtb2RlbCwgZmVhdHVyZXMsIGRpcmVjdG9yeSk6CiAgICAiIiJQcm9maWxlIEVBR0VSIEdQVSBrZXJuZWxzIHNlcGFyYXRlbHkgZnJvbSB0aW1lIGJldHdlZW4gbGF1bmNoZXMuIiIiCiAgICBmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKICAgIGltcG9ydCBqc29uCiAgICBkaXJlY3RvcnkgPSBQYXRoKGRpcmVjdG9yeSkKICAgIGRpcmVjdG9yeS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIHRvcmNoLnByb2ZpbGVyLnByb2ZpbGUoYWN0aXZpdGllcz1bdG9yY2gucHJvZmlsZXIuUHJvZmlsZXJBY3Rpdml0eS5DUFUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b3JjaC5wcm9maWxlci5Qcm9maWxlckFjdGl2aXR5LkNVREFdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlY29yZF9zaGFwZXM9VHJ1ZSkgYXMgcHJvZjoKICAgICAgICBtb2RlbChmZWF0dXJlcykKICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKGZlYXR1cmVzLmRldmljZSkKICAgIHByb2YuZXhwb3J0X2Nocm9tZV90cmFjZShzdHIoZGlyZWN0b3J5IC8gIm5ldHdvcmtfdHJhY2UuanNvbiIpKQogICAgZXZlbnRzID0gcHJvZi5rZXlfYXZlcmFnZXMoZ3JvdXBfYnlfaW5wdXRfc2hhcGU9VHJ1ZSkKICAgIChkaXJlY3RvcnkgLyAibmV0d29ya19vcGVyYXRvcnMudHh0Iikud3JpdGVfdGV4dCgKICAgICAgICBldmVudHMudGFibGUoc29ydF9ieT0ic2VsZl9kZXZpY2VfdGltZV90b3RhbCIsIHJvd19saW1pdD02MCkpCiAgICByb3dzID0gW3sib3BlcmF0b3IiOiBlLmtleSwgImNhbGxzIjogZS5jb3VudCwKICAgICAgICAgICAgICJzZWxmX2dwdV91cyI6IGUuc2VsZl9kZXZpY2VfdGltZV90b3RhbCwKICAgICAgICAgICAgICJzZWxmX2NwdV91cyI6IGUuc2VsZl9jcHVfdGltZV90b3RhbCwKICAgICAgICAgICAgICJzaGFwZXMiOiBzdHIoZS5pbnB1dF9zaGFwZXMpfSBmb3IgZSBpbiBldmVudHNdCiAgICByb3dzLnNvcnQoa2V5PWxhbWJkYSByOiByWyJzZWxmX2dwdV91cyJdLCByZXZlcnNlPVRydWUpCiAgICAoZGlyZWN0b3J5IC8gIm5ldHdvcmtfb3BlcmF0b3JzLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocm93cywgaW5kZW50PTIpKQogICAgcHJpbnQoIk9wZXJhdG9yIHByb2ZpbGU6IiwgZGlyZWN0b3J5IC8gIm5ldHdvcmtfb3BlcmF0b3JzLnR4dCIsIGZsdXNoPVRydWUpCg=='))
(FAST_RUNTIME_DIR / 'fast_temporal.py').write_bytes(base64.b64decode('IiIiQm91bmRlZCBtb3Rpb24gZ3VpZGVzIGFuZCBmdWxsLXJlc29sdXRpb24gR1BVIGZpdmUtdGFwIGhpc3Rvcnkgc2FtcGxpbmcuCgpHdWlkZSByZWR1Y3Rpb24gY2hhbmdlcyBvcHRpY2FsLWZsb3cgZXN0aW1hdGVzLCBub3QgbmV1cmFsIHJlbmRlcmluZyByZXNvbHV0aW9uLgpQYXRjaGVzIGFyZSBzY29wZWQgdG8gb25lIHNlcmlhbCBydW4gYW5kIHJlc3RvcmVkIGFmdGVyIHRoZSBwcmVmZXRjaCB3b3JrZXIgam9pbnMuCiIiIgpmcm9tIGNvbnRleHRsaWIgaW1wb3J0IGNvbnRleHRtYW5hZ2VyLCBFeGl0U3RhY2sKZnJvbSB1bml0dGVzdC5tb2NrIGltcG9ydCBwYXRjaAppbXBvcnQgdGltZQppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgoKZGVmIGd1aWRlX3NpemUod2lkdGgsIGhlaWdodCwgbWF4X3NpZGUpOgogICAgcmF0aW8gPSBtaW4oMS4wLCBtYXhfc2lkZSAvIG1heCh3aWR0aCwgaGVpZ2h0KSkKICAgIHJldHVybiBtYXgoMSwgcm91bmQod2lkdGggKiByYXRpbykpLCBtYXgoMSwgcm91bmQoaGVpZ2h0ICogcmF0aW8pKQoKCkB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpCmRlZiBncHVfaGlzdG9yeShoaXN0b3J5LCB1LCB2LCBkZXZpY2UpOgogICAgaW1hZ2UgPSB0b3JjaC5hc190ZW5zb3IobnAuYXNjb250aWd1b3VzYXJyYXkoaGlzdG9yeSksIGRldmljZT1kZXZpY2UpCiAgICB1ID0gdG9yY2guYXNfdGVuc29yKG5wLmFzY29udGlndW91c2FycmF5KHUpLCBkZXZpY2U9ZGV2aWNlKQogICAgdiA9IHRvcmNoLmFzX3RlbnNvcihucC5hc2NvbnRpZ3VvdXNhcnJheSh2KSwgZGV2aWNlPWRldmljZSkKICAgIHJldHVybiB0ZW5zb3JfaGlzdG9yeShpbWFnZSwgdSwgdikuY3B1KCkubnVtcHkoKQoKCmRlZiByb3VuZGVkX2RpdmlkZShudW1lcmF0b3IsIGRlbm9taW5hdG9yKToKICAgICIiIlJvdW5kIHRoZSBxdW90aWVudCB0byBGUDMyIHdpdGhvdXQgYSBGUDMyIHJlY2lwcm9jYWwgYXBwcm94aW1hdGlvbi4KCiAgICBIaXN0b3J5IHZhbHVlcyBhcmUgc3Vic2VxdWVudGx5IHF1YW50aXplZCB0byBoYWxmOyBhIG9uZS1VTFAgZGl2aXNpb24KICAgIGRpZmZlcmVuY2UgY2FuIGNyb3NzIHRoYXQgYm91bmRhcnkgYW5kIGJlY29tZSBhIHZpc2libGUgYmxlbmQgZGlmZmVyZW5jZS4KICAgIFRoZSB0ZW1wb3JhcnkgRlA2NCBxdW90aWVudCBpcyByb3VuZGVkIGJhY2sgaW1tZWRpYXRlbHksIG5vdCBwcm9wYWdhdGVkLgogICAgIiIiCiAgICByZXR1cm4gKG51bWVyYXRvci5kb3VibGUoKSAvIGRlbm9taW5hdG9yLmRvdWJsZSgpKS5mbG9hdCgpCgoKZGVmIHRlbnNvcl9oaXN0b3J5KGltYWdlLCB1LCB2KToKICAgICIiIlNhbWUgZml2ZSB0YXBzLCBhY2NlcHRpbmcvcmV0dXJuaW5nIHRlbnNvcnMgd2l0aG91dCBhIGhvc3Qgcm91bmQgdHJpcC4iIiIKICAgIGhlaWdodCwgd2lkdGggPSBpbWFnZS5zaGFwZVs6Ml0KICAgIGRlZiBjb29yZGluYXRlcyhub3JtYWxpemVkLCBkaW1lbnNpb24pOgogICAgICAgIHBpeGVsID0gbm9ybWFsaXplZCAqIGRpbWVuc2lvbiAtIDAuNQogICAgICAgIGJhc2VfaW5kZXggPSB0b3JjaC5mbG9vcihwaXhlbCkKICAgICAgICB0ID0gKHBpeGVsIC0gYmFzZV9pbmRleCkuY2xhbXAoMCwgMSkKICAgICAgICBzcXVhcmUsIGN1YmUgPSB0ICogdCwgdCAqIHQgKiB0CiAgICAgICAgdzAgPSAtMC41ICogdCArIHNxdWFyZSAtIDAuNSAqIGN1YmUKICAgICAgICB3MSA9IDEgLSAyLjUgKiBzcXVhcmUgKyAxLjUgKiBjdWJlCiAgICAgICAgdzIgPSAwLjUgKiB0ICsgMiAqIHNxdWFyZSAtIDEuNSAqIGN1YmUKICAgICAgICB3MyA9IC0wLjUgKiBzcXVhcmUgKyAwLjUgKiBjdWJlCiAgICAgICAgZyA9IHcxICsgdzIKICAgICAgICBiYXNlID0gYmFzZV9pbmRleCArIDAuNQogICAgICAgIHJldHVybiAoKGJhc2UgLSAxKS5jbGFtcCgwLjUsIGRpbWVuc2lvbiAtIDAuNSksCiAgICAgICAgICAgICAgICAoYmFzZSArIHJvdW5kZWRfZGl2aWRlKHcyLCBnKSkuY2xhbXAoMC41LCBkaW1lbnNpb24gLSAwLjUpLAogICAgICAgICAgICAgICAgKGJhc2UgKyAyKS5jbGFtcCgwLjUsIGRpbWVuc2lvbiAtIDAuNSksIHcwLCB3MywgZykKICAgIGRlZiBzYW1wbGUoeCwgeSk6CiAgICAgICAgcHgsIHB5ID0geCAtIDAuNSwgeSAtIDAuNQogICAgICAgIHgwID0gcHguZmxvb3IoKS5jbGFtcCgwLCB3aWR0aCAtIDEpLmxvbmcoKQogICAgICAgIHkwID0gcHkuZmxvb3IoKS5jbGFtcCgwLCBoZWlnaHQgLSAxKS5sb25nKCkKICAgICAgICB4MSwgeTEgPSAoeDAgKyAxKS5jbGFtcChtYXg9d2lkdGgtMSksICh5MCArIDEpLmNsYW1wKG1heD1oZWlnaHQtMSkKICAgICAgICB0eCwgdHkgPSAocHggLSB4MCkuY2xhbXAoMCwgMSlbLi4uLCBOb25lXSwgKHB5IC0geTApLmNsYW1wKDAsIDEpWy4uLiwgTm9uZV0KICAgICAgICB0b3AgPSBpbWFnZVt5MCwgeDBdICogKDEtdHgpICsgaW1hZ2VbeTAsIHgxXSAqIHR4CiAgICAgICAgYm90dG9tID0gaW1hZ2VbeTEsIHgwXSAqICgxLXR4KSArIGltYWdlW3kxLCB4MV0gKiB0eAogICAgICAgIHJldHVybiB0b3AgKiAoMS10eSkgKyBib3R0b20gKiB0eQogICAgeDAsIHhtLCB4MywgeHcwLCB4dzMsIHhnID0gY29vcmRpbmF0ZXModSwgd2lkdGgpCiAgICB5MCwgeW0sIHkzLCB5dzAsIHl3MywgeWcgPSBjb29yZGluYXRlcyh2LCBoZWlnaHQpCiAgICB0b3RhbCwgd2VpZ2h0X3N1bSA9IDAsIDAKICAgIGZvciB4LCB5LCB3ZWlnaHQgaW4gKCh4MCwgeW0sIHh3MCp5ZyksICh4bSwgeTAsIHhnKnl3MCksCiAgICAgICAgICAgICAgICAgICAgICAgICAgKHhtLCB5bSwgeGcqeWcpLCAoeG0sIHkzLCB4Zyp5dzMpLCAoeDMsIHltLCB4dzMqeWcpKToKICAgICAgICB0b3RhbCA9IHRvdGFsICsgd2VpZ2h0Wy4uLiwgTm9uZV0gKiBzYW1wbGUoeCwgeSkKICAgICAgICB3ZWlnaHRfc3VtID0gd2VpZ2h0X3N1bSArIHdlaWdodAogICAgcmV0dXJuIHJvdW5kZWRfZGl2aWRlKHRvdGFsLCB3ZWlnaHRfc3VtWy4uLiwgTm9uZV0pCgoKZGVmIHZhbGlkYXRlX2hpc3RvcnkoZGV2aWNlKToKICAgIGZyb20gbWx4ZGxzcy50ZW1wb3JhbCBpbXBvcnQgc2FtcGxlX2hpc3RvcnkKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyg3MykKICAgIHdvcnN0ID0gMC4wCiAgICBmb3IgaGVpZ2h0LCB3aWR0aCBpbiAoKDEsIDEpLCAoMzEsIDQ3KSwgKDczLCA5NikpOgogICAgICAgIGhpc3RvcnkgPSBybmcucmFuZG9tKChoZWlnaHQsIHdpZHRoLCAzKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICB5eSwgeHggPSBucC5pbmRpY2VzKChoZWlnaHQsIHdpZHRoKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICBmb3Igb2Zmc2V0IGluICgwLjAsIDAuMDMyLCAtMC4xKToKICAgICAgICAgICAgdSwgdiA9ICh4eCswLjUpL3dpZHRoICsgb2Zmc2V0LCAoeXkrMC41KS9oZWlnaHQgLSBvZmZzZXQKICAgICAgICAgICAgZXhwZWN0ZWQgPSBzYW1wbGVfaGlzdG9yeShoaXN0b3J5LCB1LCB2KQogICAgICAgICAgICBhY3R1YWwgPSBncHVfaGlzdG9yeShoaXN0b3J5LCB1LCB2LCBkZXZpY2UpCiAgICAgICAgICAgIGVycm9yID0gZmxvYXQobnAubWF4KG5wLmFicyhleHBlY3RlZCAtIGFjdHVhbCkpKQogICAgICAgICAgICB3b3JzdCA9IG1heCh3b3JzdCwgZXJyb3IpCiAgICAgICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShhY3R1YWwpLmFsbCgpIG9yIGVycm9yID4gMmUtNToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkdQVSBoaXN0b3J5IHNhbXBsZXIgcGFyaXR5IGZhaWxlZDogbWF4IGVycm9yIHtlcnJvcn0iKQogICAgcmV0dXJuIHsibWF4X2Fic29sdXRlX2Vycm9yIjogd29yc3QsICJwYXNzZWQiOiBUcnVlLCAidG9sZXJhbmNlIjogMmUtNX0KCgpAY29udGV4dG1hbmFnZXIKZGVmIHRlbXBvcmFsX3J1bnRpbWUoKiwgbWF4X3NpZGUsIGRldmljZSwgdXNlX2dwdV9oaXN0b3J5LCBzdGF0cywgcmVzaWRlbnQ9RmFsc2UpOgogICAgZnJvbSBtbHhkbHNzIGltcG9ydCB0ZW1wb3JhbCBhcyB0CiAgICBpbXBvcnQgY3YyCiAgICBpZiBtYXhfc2lkZSA8IDY0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk1vdGlvbi1ndWlkZSBsb25nZXN0IHNpZGUgbXVzdCBiZSBhdCBsZWFzdCA2NCIpCiAgICBvcmlnaW5hbF9jbGFzcyA9IHQuRmxvd01vdGlvbkVzdGltYXRvcgogICAgY2xhc3MgQm91bmRlZEZsb3cob3JpZ2luYWxfY2xhc3MpOgogICAgICAgIGRlZiBlc3RpbWF0ZShzZWxmLCBjdXJyZW50LCBwcmV2aW91cywgKiwgc2NlbmVfY3V0X3RocmVzaG9sZD0wLjMpOgogICAgICAgICAgICBoLCB3ID0gY3VycmVudC5zaGFwZVs6Ml0KICAgICAgICAgICAgZ3csIGdoID0gZ3VpZGVfc2l6ZSh3LCBoLCBtYXhfc2lkZSkKICAgICAgICAgICAgaWYgKGd3LCBnaCkgPT0gKHcsIGgpOgogICAgICAgICAgICAgICAgcmV0dXJuIHN1cGVyKCkuZXN0aW1hdGUoY3VycmVudCwgcHJldmlvdXMsIHNjZW5lX2N1dF90aHJlc2hvbGQ9c2NlbmVfY3V0X3RocmVzaG9sZCkKICAgICAgICAgICAgc21hbGxfY3VycmVudCA9IGN2Mi5yZXNpemUoY3VycmVudCwgKGd3LCBnaCksIGludGVycG9sYXRpb249Y3YyLklOVEVSX0FSRUEpCiAgICAgICAgICAgIHNtYWxsX3ByZXZpb3VzID0gY3YyLnJlc2l6ZShwcmV2aW91cywgKGd3LCBnaCksIGludGVycG9sYXRpb249Y3YyLklOVEVSX0FSRUEpCiAgICAgICAgICAgIGVzdGltYXRlID0gc3VwZXIoKS5lc3RpbWF0ZShzbWFsbF9jdXJyZW50LCBzbWFsbF9wcmV2aW91cywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNjZW5lX2N1dF90aHJlc2hvbGQ9c2NlbmVfY3V0X3RocmVzaG9sZCkKICAgICAgICAgICAgIyBNb3Rpb24gYWxyZWFkeSB1c2VzIG5vcm1hbGl6ZWQgVVYgdW5pdHM6IGRvIE5PVCBtdWx0aXBseSBieSBzY2FsZS4KICAgICAgICAgICAgZXN0aW1hdGUubW90aW9uX3V2ID0gY3YyLnJlc2l6ZShlc3RpbWF0ZS5tb3Rpb25fdXYsICh3LCBoKSwgaW50ZXJwb2xhdGlvbj1jdjIuSU5URVJfTElORUFSKQogICAgICAgICAgICBlc3RpbWF0ZS5jb25maWRlbmNlID0gY3YyLnJlc2l6ZShlc3RpbWF0ZS5jb25maWRlbmNlLCAodywgaCksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpbnRlcnBvbGF0aW9uPWN2Mi5JTlRFUl9ORUFSRVNUKVsuLi4sIE5vbmVdCiAgICAgICAgICAgIHJldHVybiBlc3RpbWF0ZQoKICAgIGRlZiB0aW1lZChuYW1lLCBmdW5jdGlvbik6CiAgICAgICAgZGVmIGNhbGwoKmFyZ3MsICoqa3dhcmdzKToKICAgICAgICAgICAgc3RhcnQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHJldHVybiBmdW5jdGlvbigqYXJncywgKiprd2FyZ3MpCiAgICAgICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgICAgICBzdGF0c1tuYW1lXSA9IHN0YXRzLmdldChuYW1lLCAwLjApICsgdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0CiAgICAgICAgcmV0dXJuIGNhbGwKICAgIHdpdGggRXhpdFN0YWNrKCkgYXMgc3RhY2s6CiAgICAgICAgc3RhY2suZW50ZXJfY29udGV4dChwYXRjaC5vYmplY3QodCwgIkZsb3dNb3Rpb25Fc3RpbWF0b3IiLCBCb3VuZGVkRmxvdykpCiAgICAgICAgaWYgdXNlX2dwdV9oaXN0b3J5OgogICAgICAgICAgICBzdGFjay5lbnRlcl9jb250ZXh0KHBhdGNoLm9iamVjdCh0LCAic2FtcGxlX2hpc3RvcnkiLAogICAgICAgICAgICAgICAgbGFtYmRhIGhpc3RvcnksIHUsIHY6IGdwdV9oaXN0b3J5KGhpc3RvcnksIHUsIHYsIGRldmljZSkpKQogICAgICAgICMgSW5jbHVzaXZlIHN0YWdlIHRpbWVzOiBoaXN0b3J5IHNhbXBsaW5nIGlzIHBhcnQgb2YgdGVtcG9yYWxfZmVhdHVyZXMuCiAgICAgICAgZm9yIG5hbWUgaW4gKCJzYW1wbGVfaGlzdG9yeSIsICJtYWtlX2ZlYXR1cmVzIiwgIm1ha2VfdGVtcG9yYWxfZmVhdHVyZXMiLCAiZXh0ZW5kX2ZlYXR1cmVzIiwKICAgICAgICAgICAgICAgICAgICAgImNvbXBvc2VfaGVhZCIsICJjb21wb3NlX3RlbXBvcmFsIiwgImNvbXBvc2VfZGV0YWlsIiwgInJlc29sdmVfbW90aW9uIiwKICAgICAgICAgICAgICAgICAgICAgInByZXBhcmVfdGVtcG9yYWxfZnJhbWUiKToKICAgICAgICAgICAgc3RhY2suZW50ZXJfY29udGV4dChwYXRjaC5vYmplY3QodCwgbmFtZSwgdGltZWQobmFtZSwgZ2V0YXR0cih0LCBuYW1lKSkpKQogICAgICAgICMgVmlkZW8gc3RyZWFtaW5nIGltcG9ydHMgdGhlc2UgdHdvIG5hbWVzIGRpcmVjdGx5LCBpbmNsdWRpbmcgaXRzCiAgICAgICAgIyBDUFUtb25seSBtb3Rpb24gcHJlZmV0Y2ggcGF0aDsgY292ZXIgYm90aCBpdCBhbmQgc2Vzc2lvbi5wcm9jZXNzKCkuCiAgICAgICAgZnJvbSBtbHhkbHNzIGltcG9ydCB2aWRlb19waXBlbGluZQogICAgICAgIGZvciBuYW1lIGluICgicmVzb2x2ZV9tb3Rpb24iLCAicHJlcGFyZV90ZW1wb3JhbF9mcmFtZSIpOgogICAgICAgICAgICBzdGFjay5lbnRlcl9jb250ZXh0KHBhdGNoLm9iamVjdCh2aWRlb19waXBlbGluZSwgbmFtZSwgZ2V0YXR0cih0LCBuYW1lKSkpCiAgICAgICAgaWYgcmVzaWRlbnQ6CiAgICAgICAgICAgIGZyb20gcmVzaWRlbnRfdGVtcG9yYWwgaW1wb3J0IHJlc2lkZW50X3Nlc3Npb25fY2xhc3MKICAgICAgICAgICAgc2Vzc2lvbl9jbGFzcyA9IHJlc2lkZW50X3Nlc3Npb25fY2xhc3ModC5UZW1wb3JhbFNlc3Npb24sIHN0YXRzKQogICAgICAgICAgICBzdGFjay5lbnRlcl9jb250ZXh0KHBhdGNoLm9iamVjdCh0LCAiVGVtcG9yYWxTZXNzaW9uIiwgc2Vzc2lvbl9jbGFzcykpCiAgICAgICAgICAgIHN0YWNrLmVudGVyX2NvbnRleHQocGF0Y2gub2JqZWN0KHZpZGVvX3BpcGVsaW5lLCAiVGVtcG9yYWxTZXNzaW9uIiwgc2Vzc2lvbl9jbGFzcykpCiAgICAgICAgeWllbGQK'))
(FAST_RUNTIME_DIR / 'resident_temporal.py').write_bytes(base64.b64decode('IiIiR1BVIGZlYXR1cmUgY29uc3RydWN0aW9uLCBoaXN0b3J5IGFuZCBjb21wb3NpdGlvbiBmb3IgdGhlIHBpbm5lZCB0ZW1wb3JhbCBwYXRoLgoKQ1BVIG1vdGlvbi9MYW5jem9zIGlucHV0IHJlc2l6ZSBpcyByZXRhaW5lZCBpbiB0aGUgZXhpc3RpbmcgcHJlZmV0Y2ggd29ya2VyLgpObyBob3N0LXNpemVkIHNpeHRlZW4tY2hhbm5lbCBhcnJheXMgYXJlIGNvbnN0cnVjdGVkIGluIHRoaXMgc2Vzc2lvbi4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKaW1wb3J0IHRpbWUKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIGZhc3RfdGVtcG9yYWwgaW1wb3J0IHRlbnNvcl9oaXN0b3J5CgoKZGVmIGhhbGYoeCk6CiAgICByZXR1cm4geC50byh0b3JjaC5mbG9hdDE2KS50byh0b3JjaC5mbG9hdDMyKQoKCmRlZiBzY2FsZWRfY29sb3IoY29sb3IpOgogICAgcmV0dXJuIGhhbGYoaGFsZihoYWxmKGNvbG9yKSAtIDAuNSkgKiAwLjEyNSkKCgpkZWYgc2hpZnRfbWl4KHZhbHVlKToKICAgIHZhbHVlID0gdmFsdWUgJiAweEZGRkZGRkZGCiAgICBtaXhlZCA9IHZhbHVlIF4gKHZhbHVlID4+ICgodmFsdWUgPj4gMjgpICsgNCkpCiAgICByZXR1cm4gKG1peGVkICogMHgxMDhFRjJEOSkgJiAweEZGRkZGRkZGCgoKZGVmIHVuaWZvcm0yNCh2YWx1ZSk6CiAgICBtaXhlZCA9IHNoaWZ0X21peCh2YWx1ZSkKICAgIGJpdHMgPSAobWl4ZWQgPj4gMzApIF4gKG1peGVkID4+IDgpCiAgICByZXR1cm4gKGJpdHMgKyAxKS5mbG9hdCgpICogNS45NjA0NjQ0Nzc1MzkwNjNlLTgKCgpkZWYgbm9pc2VfZnJvbV9zZWVkKHNwYXRpYWxfc2VlZCwgZnJhbWVfaW5kZXgpOgogICAgc2VlZCA9IHNwYXRpYWxfc2VlZCBeICgoaW50KGZyYW1lX2luZGV4KSAqIDB4OUUzNzc5QjkpICYgMHhGRkZGRkZGRikKICAgIG11bHRpcGxpZWQgPSBzaGlmdF9taXgoc2VlZCkKICAgIG1peGVkID0gbXVsdGlwbGllZCBeIChtdWx0aXBsaWVkID4+IDIyKQogICAgdWEgPSB1bmlmb3JtMjQobWl4ZWQgKiAweENBQTVCODBEICsgMHgyMURENzk2QikKICAgIHViID0gdW5pZm9ybTI0KG1peGVkICogMHg4MzIzMkMzMSArIDB4MzQ2M0UwQUMpCiAgICB1YyA9IHVuaWZvcm0yNChtaXhlZCAqIDB4MkM5Mjc3QjUgKyAweEFDNTY0QjA1KQogICAgdWQgPSB1bmlmb3JtMjQobWl4ZWQgKiAweEZBNkRDNUY5ICsgMHg0NzEyQTg4RSkKICAgIHJhLCByYiA9ICgtMiAqIHVhLmxvZygpKS5zcXJ0KCksICgtMiAqIHVjLmxvZygpKS5zcXJ0KCkKICAgIGFhLCBhYiA9IDYuMjgzMTg1NDgyMDI1MTQ2NSAqIHVkLCA2LjI4MzE4NTQ4MjAyNTE0NjUgKiB1YgogICAgcmV0dXJuIHRvcmNoLnN0YWNrKChyYiAqIGFhLmNvcygpLCByYiAqIGFhLnNpbigpLCByYSAqIGFiLmNvcygpKSwgLTEpLmhhbGYoKQoKCmNsYXNzIEZlYXR1cmVCdWlsZGVyOgogICAgIiIiT25lIGZpeGVkIHNoYXBlOyBjYWNoZXMgY29vcmRpbmF0ZXMsIHNlZWQgYW5kIHJldXNhYmxlIGhhbGYgaW5wdXQgc3RvcmFnZS4iIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBnZW9tZXRyeSwgZGV2aWNlKToKICAgICAgICBzZWxmLmdlb21ldHJ5ID0gZ2VvbWV0cnkKICAgICAgICBoLCB3ID0gZ2VvbWV0cnkub3V0cHV0X2hlaWdodCwgZ2VvbWV0cnkub3V0cHV0X3dpZHRoCiAgICAgICAgbmgsIG53ID0gZ2VvbWV0cnkubmV0d29ya19oZWlnaHQsIGdlb21ldHJ5Lm5ldHdvcmtfd2lkdGgKICAgICAgICB5ID0gdG9yY2guYXJhbmdlKG5oLCBkZXZpY2U9ZGV2aWNlLCBkdHlwZT10b3JjaC5pbnQ2NClbOiwgTm9uZV0KICAgICAgICB4ID0gdG9yY2guYXJhbmdlKG53LCBkZXZpY2U9ZGV2aWNlLCBkdHlwZT10b3JjaC5pbnQ2NClbTm9uZSwgOl0KICAgICAgICBzZWxmLnNlZWQgPSAoKHkgKiAweEQ4MTYzODQxKSBeICh4ICogMHg4REE2QjM0MykgXiAweDI0M0Y2QTg4KSAmIDB4RkZGRkZGRkYKICAgICAgICBzZWxmLnJvd3MgPSB0b3JjaC53aGVyZSh5IDwgaCwgeSwgKDIqaC0yLXkpLmNsYW1wKG1pbj0wKSkKICAgICAgICBzZWxmLmNvbHMgPSB0b3JjaC53aGVyZSh4IDwgdywgeCwgKDIqdy0yLXgpLmNsYW1wKG1pbj0wKSkKICAgICAgICAjIEJ1aWxkIHRoZXNlIHNtYWxsIGNhY2hlZCBheGVzIHdpdGggdGhlIGV4YWN0IHJlZmVyZW5jZSBhcml0aG1ldGljLgogICAgICAgICMgQ1VEQSBzY2FsYXIgZGl2aXNpb24gbWF5IGluc3RlYWQgbXVsdGlwbHkgYnkgYW4gYXBwcm94aW1hdGUgcmVjaXByb2NhbDsKICAgICAgICAjIGEgdGlueSBVViBlcnJvciBjYW4gY3Jvc3MgYSBsYXRlciBGUDE2IGhpc3Rvcnktcm91bmRpbmcgYm91bmRhcnkuCiAgICAgICAgc2VsZi51ID0gdG9yY2guYXNfdGVuc29yKChucC5hcmFuZ2UodywgZHR5cGU9bnAuZmxvYXQzMilbTm9uZSwgOl0gKyBucC5mbG9hdDMyKDAuNSkpIC8gbnAuZmxvYXQzMih3KSwgZGV2aWNlPWRldmljZSkKICAgICAgICBzZWxmLnYgPSB0b3JjaC5hc190ZW5zb3IoKG5wLmFyYW5nZShoLCBkdHlwZT1ucC5mbG9hdDMyKVs6LCBOb25lXSArIG5wLmZsb2F0MzIoMC41KSkgLyBucC5mbG9hdDMyKGgpLCBkZXZpY2U9ZGV2aWNlKQogICAgICAgIHNlbGYuZmVhdHVyZXMgPSB0b3JjaC5lbXB0eSgoMSwgbmgsIG53LCAxNiksIGR0eXBlPXRvcmNoLmZsb2F0MTYsIGRldmljZT1kZXZpY2UpCgogICAgZGVmIGJ1aWxkKHNlbGYsIGNvbG9yLCBmcmFtZV9pbmRleCwgY29udHJvbHMsICosIGhpc3Rvcnk9Tm9uZSwgbW90aW9uPU5vbmUsIGNvbmZpZGVuY2U9Tm9uZSk6CiAgICAgICAgY3VycmVudCA9IHNjYWxlZF9jb2xvcihjb2xvcikKICAgICAgICBoaXN0b3J5X2ZlYXR1cmVzID0gY3VycmVudAogICAgICAgIGlmIGhpc3RvcnkgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJlcHJvamVjdGVkID0gc2NhbGVkX2NvbG9yKHRlbnNvcl9oaXN0b3J5KGhpc3RvcnksIHNlbGYudSArIG1vdGlvblsuLi4sIDBdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi52ICsgbW90aW9uWy4uLiwgMV0pKQogICAgICAgICAgICBpZiBjb25maWRlbmNlIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBoaXN0b3J5X2ZlYXR1cmVzID0gcmVwcm9qZWN0ZWQKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIG1peGVkID0gY3VycmVudCArIGNvbmZpZGVuY2UgKiAocmVwcm9qZWN0ZWQgLSBjdXJyZW50KQogICAgICAgICAgICAgICAgaGlzdG9yeV9mZWF0dXJlcyA9IHRvcmNoLndoZXJlKGNvbmZpZGVuY2UgPT0gMCwgY3VycmVudCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdG9yY2gud2hlcmUoY29uZmlkZW5jZSA9PSAxLCByZXByb2plY3RlZCwgbWl4ZWQpKQogICAgICAgIGYgPSBzZWxmLmZlYXR1cmVzWzBdCiAgICAgICAgZlsuLi4sIDozXSA9IG5vaXNlX2Zyb21fc2VlZChzZWxmLnNlZWQsIGZyYW1lX2luZGV4KQogICAgICAgIGZbLi4uLCAzXSA9IDEKICAgICAgICBmWy4uLiwgNDo3XSA9IGN1cnJlbnRbc2VsZi5yb3dzLCBzZWxmLmNvbHNdCiAgICAgICAgZlsuLi4sIDc6MTBdID0gaGlzdG9yeV9mZWF0dXJlc1tzZWxmLnJvd3MsIHNlbGYuY29sc10KICAgICAgICBmWy4uLiwgMTBdID0gY29udHJvbHNbJ25vcm1hbGl6ZWRfc3R5bGUnXQogICAgICAgIGZbLi4uLCAxMV0gPSBjb250cm9sc1snbG9jYWxfdG9uZV9zdHJlbmd0aCddCiAgICAgICAgZlsuLi4sIDEyXSA9IGNvbnRyb2xzWydsb2NhbF9zdHJ1Y3R1cmVfc3RyZW5ndGgnXQogICAgICAgIGZbLi4uLCAxMzoxNV0gPSAtMQogICAgICAgIGZbLi4uLCAxNV0gPSAwCiAgICAgICAgIyBLZWVwIEZMT0FUMzIgbWl4ZWQgaGlzdG9yeSBmb3IgY29tcG9zaXRpb246IGhhbGYgaW5wdXQgcm91bmRpbmcgbXVzdAogICAgICAgICMgbm90IGxlYWsgaW50byB0aGUgb3JpZ2luYWwgYmxlbmQgZm9ybXVsYS4KICAgICAgICByZXR1cm4gc2VsZi5mZWF0dXJlcywgaGlzdG9yeV9mZWF0dXJlcwoKCmRlZiBjb21wb3NlKGhlYWQsIGNvbG9yLCBoaXN0b3J5X2ZlYXR1cmVzLCBjb25maWRlbmNlLCAqLCB0ZW1wb3JhbCwgaW50ZW5zaXR5LCBibGVuZF9zY2FsZSk6CiAgICBoZWFkID0gaGVhZC5mbG9hdCgpCiAgICBwcmVkaWN0ZWQgPSAoY29sb3IgKyBoYWxmKGhlYWRbLi4uLCA6M10pICogMC4yNSkuY2xhbXAoMCwgMSkKICAgIGlmIHRlbXBvcmFsOgogICAgICAgIGxvZ2l0ID0gaGFsZihoZWFkWy4uLiwgMzo0XSkKICAgICAgICAjIEV4cGxpY2l0IG9wZXJhdGlvbnMgcHJlc2VydmUgdGhlIHJlZmVyZW5jZSdzIHNpZ21vaWQgYXJpdGhtZXRpYy4KICAgICAgICByb3VuZGVkX3NjYWxlID0gZmxvYXQobnAuZmxvYXQxNihibGVuZF9zY2FsZSkpCiAgICAgICAgYWxwaGEgPSAoKDEgLyAoMSArIHRvcmNoLmV4cCgtbG9naXQpKSkgKiByb3VuZGVkX3NjYWxlKS5jbGFtcCgwLCAxKQogICAgICAgIGlmIGNvbmZpZGVuY2UgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGFscGhhID0gYWxwaGEgKiBjb25maWRlbmNlCiAgICAgICAgcmVjb25zdHJ1Y3RlZCA9IGhpc3RvcnlfZmVhdHVyZXMgKiA4ICsgMC41CiAgICAgICAgb3V0cHV0ID0gcHJlZGljdGVkICsgYWxwaGEgKiAocmVjb25zdHJ1Y3RlZCAtIHByZWRpY3RlZCkKICAgICAgICBpZiBpbnRlbnNpdHkgPT0gMToKICAgICAgICAgICAgcmV0dXJuIG91dHB1dAogICAgZWxzZToKICAgICAgICBvdXRwdXQgPSBwcmVkaWN0ZWQKICAgIHN0cmVuZ3RoID0gZmxvYXQobnAuY2xpcChucC5mbG9hdDMyKGludGVuc2l0eSksIDAsIDEpKQogICAgcmV0dXJuIChjb2xvciArIHN0cmVuZ3RoICogKG91dHB1dCAtIGNvbG9yKSkuY2xhbXAoMCwgMSkKCgpkZWYgaW50ZWdlcl9kb3duc2FtcGxlKG91dHB1dCwgaGVpZ2h0LCB3aWR0aCk6CiAgICBoLCB3ID0gb3V0cHV0LnNoYXBlWzoyXQogICAgaWYgKGgsIHcpID09IChoZWlnaHQsIHdpZHRoKToKICAgICAgICByZXR1cm4gb3V0cHV0CiAgICBpZiBoICUgaGVpZ2h0IG9yIHcgJSB3aWR0aCBvciBoIC8vIGhlaWdodCAhPSB3IC8vIHdpZHRoOgogICAgICAgIHJldHVybiBOb25lICAjIHJldGFpbiB1cHN0cmVhbSBMYW5jem9zIGZvciBub24taW50ZWdlciBwcm9jZXNzaW5nIHNjYWxlcwogICAgZmFjdG9yID0gaCAvLyBoZWlnaHQKICAgIHJldHVybiBvdXRwdXQucmVzaGFwZShoZWlnaHQsIGZhY3Rvciwgd2lkdGgsIGZhY3RvciwgMykubWVhbihkaW09KDEsIDMpKQoKCmRlZiByZXNpZGVudF9zZXNzaW9uX2NsYXNzKGJhc2UsIHN0YXRzKToKICAgIGNsYXNzIFJlc2lkZW50U2Vzc2lvbihiYXNlKToKICAgICAgICBAdG9yY2guaW5mZXJlbmNlX21vZGUoKQogICAgICAgIGRlZiBfcHJvY2Vzc19wcmVwYXJlZChzZWxmLCBwcmVwYXJlZCwgY29udHJvbF9tYXNrPU5vbmUpOgogICAgICAgICAgICBmcm9tIG1seGRsc3MuZmVhdHVyZXMgaW1wb3J0IE5ldHdvcmtHZW9tZXRyeQogICAgICAgICAgICBmcm9tIG1seGRsc3MuY29tcG9zaXRpb24gaW1wb3J0IHJlc2FtcGxlLCBjb21wb3NlX2RldGFpbAogICAgICAgICAgICBpZiBjb250cm9sX21hc2sgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdSZXNpZGVudCBwYXRoIGRvZXMgbm90IHN1cHBvcnQgY29udHJvbCBtYXNrczsgZGlzYWJsZSBHUFVfUkVTSURFTlRfVEVNUE9SQUwnKQogICAgICAgICAgICBpZiBwcmVwYXJlZC5yZXNldDoKICAgICAgICAgICAgICAgIHNlbGYucmVzZXQoKQogICAgICAgICAgICAgICAgc2VsZi5zY2VuZV9jdXRzICs9IDEKICAgICAgICAgICAgaWYgc2VsZi5oaXN0b3J5IGlzIG5vdCBOb25lIGFuZCB0dXBsZShzZWxmLmhpc3Rvcnkuc2hhcGUpICE9IHByZXBhcmVkLmNvbG9yLnNoYXBlOgogICAgICAgICAgICAgICAgc2VsZi5yZXNldCgpCiAgICAgICAgICAgIHN0YXJ0ZWQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGV2ZW50cyA9IFtdCiAgICAgICAgICAgIGRlZiBtYXJrKG5hbWUsIGZuKToKICAgICAgICAgICAgICAgIGEsIGIgPSBbdG9yY2guY3VkYS5FdmVudChlbmFibGVfdGltaW5nPVRydWUpIGZvciBfIGluIHJhbmdlKDIpXQogICAgICAgICAgICAgICAgYS5yZWNvcmQoKQogICAgICAgICAgICAgICAgcmVzdWx0ID0gZm4oKQogICAgICAgICAgICAgICAgYi5yZWNvcmQoKQogICAgICAgICAgICAgICAgZXZlbnRzLmFwcGVuZCgobmFtZSwgYSwgYikpCiAgICAgICAgICAgICAgICByZXR1cm4gcmVzdWx0CiAgICAgICAgICAgIHdpdGggdG9yY2guY3VkYS5kZXZpY2Uoc2VsZi5waXBlbGluZS5kZXZpY2UpOgogICAgICAgICAgICAgICAgZGVmIHVwbG9hZCgpOgogICAgICAgICAgICAgICAgICAgIGRlZiB0ZW5zb3IodmFsdWUpOgogICAgICAgICAgICAgICAgICAgICAgICByZXR1cm4gdG9yY2guYXNfdGVuc29yKG5wLmFzY29udGlndW91c2FycmF5KHZhbHVlKSwgZGV2aWNlPXNlbGYucGlwZWxpbmUuZGV2aWNlKQogICAgICAgICAgICAgICAgICAgIHJldHVybiAodGVuc29yKHByZXBhcmVkLmNvbG9yKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIE5vbmUgaWYgcHJlcGFyZWQuY29uZmlkZW5jZSBpcyBOb25lIGVsc2UgdGVuc29yKHByZXBhcmVkLmNvbmZpZGVuY2UpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgTm9uZSBpZiBzZWxmLmhpc3RvcnkgaXMgTm9uZSBlbHNlIHRlbnNvcihwcmVwYXJlZC5tb3Rpb24pKQogICAgICAgICAgICAgICAgY29sb3IsIGNvbmZpZGVuY2UsIG1vdGlvbiA9IG1hcmsoJ3Jlc2lkZW50X3VwbG9hZF9ncHVfc2Vjb25kcycsIHVwbG9hZCkKICAgICAgICAgICAgICAgIGgsIHcgPSBjb2xvci5zaGFwZVs6Ml0KICAgICAgICAgICAgICAgIGdlb21ldHJ5ID0gTmV0d29ya0dlb21ldHJ5LnZlbmRvcl9hbGlnbmVkKHcsIGgpCiAgICAgICAgICAgICAgICBpZiBub3QgaGFzYXR0cihzZWxmLCAnX2J1aWxkZXInKSBvciBzZWxmLl9idWlsZGVyLmdlb21ldHJ5ICE9IGdlb21ldHJ5OgogICAgICAgICAgICAgICAgICAgIHNlbGYuX2J1aWxkZXIgPSBGZWF0dXJlQnVpbGRlcihnZW9tZXRyeSwgc2VsZi5waXBlbGluZS5kZXZpY2UpCiAgICAgICAgICAgICAgICBoYWRfaGlzdG9yeSA9IHNlbGYuaGlzdG9yeSBpcyBub3QgTm9uZQogICAgICAgICAgICAgICAgbmV0d29yaywgaGlzdG9yeV9mZWF0dXJlcyA9IG1hcmsoJ3Jlc2lkZW50X2ZlYXR1cmVzX2dwdV9zZWNvbmRzJywgbGFtYmRhOgogICAgICAgICAgICAgICAgICAgIHNlbGYuX2J1aWxkZXIuYnVpbGQoY29sb3IsIHNlbGYuZnJhbWVfaW5kZXgsIHNlbGYuX2NvbnRyb2xzKCksIGhpc3Rvcnk9c2VsZi5oaXN0b3J5LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbW90aW9uPW1vdGlvbiwgY29uZmlkZW5jZT1jb25maWRlbmNlKSkKICAgICAgICAgICAgICAgIGhlYWQgPSBzZWxmLnBpcGVsaW5lLnJ1bl9kZXZpY2VfZmVhdHVyZXMobmV0d29yaylbMCwgOmgsIDp3XQogICAgICAgICAgICAgICAgb3V0cHV0ID0gbWFyaygncmVzaWRlbnRfY29tcG9zZV9ncHVfc2Vjb25kcycsIGxhbWJkYTogY29tcG9zZSgKICAgICAgICAgICAgICAgICAgICBoZWFkLCBjb2xvciwgaGlzdG9yeV9mZWF0dXJlcywgY29uZmlkZW5jZSwgdGVtcG9yYWw9aGFkX2hpc3RvcnksCiAgICAgICAgICAgICAgICAgICAgaW50ZW5zaXR5PXNlbGYub3B0aW9ucy5pbnRlbnNpdHksIGJsZW5kX3NjYWxlPXNlbGYub3B0aW9ucy5ibGVuZF9zY2FsZSkpCiAgICAgICAgICAgICAgICAjIE91dHB1dCBoYXMgaXRzIG93biBzdG9yYWdlLCBub3QgdGhlIHJlcGxheSBncmFwaCdzIHJldXNlZCBoZWFkLgogICAgICAgICAgICAgICAgc2VsZi5oaXN0b3J5ID0gb3V0cHV0CiAgICAgICAgICAgICAgICBzZWxmLnByZXZpb3VzID0gcHJlcGFyZWQuc291cmNlICAjIGZyYW1lcyBhcmUgaW1tdXRhYmxlIGluIHVwc3RyZWFtIHByZWZldGNoCiAgICAgICAgICAgICAgICBzZWxmLmZyYW1lX2luZGV4ICs9IDEKICAgICAgICAgICAgICAgIHNoLCBzdyA9IHByZXBhcmVkLnNvdXJjZS5zaGFwZVs6Ml0KICAgICAgICAgICAgICAgIHNtYWxsID0gbWFyaygncmVzaWRlbnRfcmVzaXplX2dwdV9zZWNvbmRzJywgbGFtYmRhOiBpbnRlZ2VyX2Rvd25zYW1wbGUob3V0cHV0LCBzaCwgc3cpKQogICAgICAgICAgICAgICAgZG93bmxvYWRfc3RhcnQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgICAgICBpZiBzbWFsbCBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgIHJlc3VsdCA9IHJlc2FtcGxlKG91dHB1dC5jcHUoKS5udW1weSgpLCBzdywgc2gpCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIHJlc3VsdCA9IHNtYWxsLmNwdSgpLm51bXB5KCkKICAgICAgICAgICAgICAgIHN0YXRzWydyZXNpZGVudF9kb3dubG9hZF9hbmRfZmFsbGJhY2tfc2Vjb25kcyddID0gc3RhdHMuZ2V0KAogICAgICAgICAgICAgICAgICAgICdyZXNpZGVudF9kb3dubG9hZF9hbmRfZmFsbGJhY2tfc2Vjb25kcycsIDAuMCkgKyB0aW1lLnBlcmZfY291bnRlcigpIC0gZG93bmxvYWRfc3RhcnQKICAgICAgICAgICAgICAgICMgSG9zdCBjb3B5IHN5bmNocm9uaXplZCBhbGwgcXVldWVkIHdvcmsuIEV2ZW50IHRpbWVzIGJlbG93IGFyZQogICAgICAgICAgICAgICAgIyBHUFUgaW50ZXJ2YWxzLCB3aGlsZSBzZXNzaW9uIHdhbGwgaW5jbHVkZXMgbGF1bmNoZXMgYW5kIHdhaXRzLgogICAgICAgICAgICAgICAgZm9yIG5hbWUsIGEsIGIgaW4gZXZlbnRzOgogICAgICAgICAgICAgICAgICAgIHN0YXRzW25hbWVdID0gc3RhdHMuZ2V0KG5hbWUsIDAuMCkgKyBhLmVsYXBzZWRfdGltZShiKSAvIDEwMDAKICAgICAgICAgICAgICAgIHJlc3VsdCA9IGNvbXBvc2VfZGV0YWlsKHByZXBhcmVkLnNvdXJjZSwgcmVzdWx0LCBkZXRhaWxfc3RyZW5ndGg9c2VsZi5vcHRpb25zLmRldGFpbF9zdHJlbmd0aCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNvbG91cl9zdHJlbmd0aD1zZWxmLm9wdGlvbnMuY29sb3VyX3N0cmVuZ3RoLCByYWRpdXM9c2VsZi5vcHRpb25zLmRldGFpbF9yYWRpdXMpCiAgICAgICAgICAgICAgICBpZiBub3QgbnAuaXNmaW5pdGUocmVzdWx0KS5hbGwoKToKICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ05vbmZpbml0ZSByZXNpZGVudCBvdXRwdXQ7IHJlZnVzaW5nIGV4cG9ydCcpCiAgICAgICAgICAgICAgICBzdGF0c1sncmVzaWRlbnRfc2Vzc2lvbl93YWxsX3NlY29uZHMnXSA9IHN0YXRzLmdldCgncmVzaWRlbnRfc2Vzc2lvbl93YWxsX3NlY29uZHMnLCAwLjApICsgdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQKICAgICAgICAgICAgICAgIHJldHVybiByZXN1bHQKICAgIHJldHVybiBSZXNpZGVudFNlc3Npb24KCgpAdG9yY2guaW5mZXJlbmNlX21vZGUoKQpkZWYgdmFsaWRhdGVfcmVzaWRlbnQoZGV2aWNlKToKICAgICIiIkdQVSBudW1lcmljIHNlbGYtY2hlY2tzOyBjb21wbGV0ZSByZWFsLXZpZGVvIHBhcml0eSByZW1haW5zIG1hbmRhdG9yeS4iIiIKICAgIGZyb20gbWx4ZGxzcyBpbXBvcnQgZmVhdHVyZXMgYXMgZiwgdGVtcG9yYWwgYXMgdAogICAgZnJvbSBtbHhkbHNzLmNvbXBvc2l0aW9uIGltcG9ydCBjb21wb3NlX2hlYWQsIHJlc2FtcGxlCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoMTIpCiAgICByZXBvcnQgPSB7J3Bhc3NlZCc6IEZhbHNlLCAnbm9pc2VfbWF4JzogMC4wLCAnbm9pc2VfbWFlJzogMC4wLAogICAgICAgICAgICAgICdmZWF0dXJlX25vbm5vaXNlX21heCc6IDAuMCwgJ2NvbXBvc2l0aW9uX21heCc6IDAuMCwgJ3Jlc2l6ZV9tYXgnOiAwLjAsCiAgICAgICAgICAgICAgJ2hpc3RvcnlfZmVhdHVyZXNfbWF4JzogMC4wLCAnY29tcG9zaXRpb25fc2FtZV9oaXN0b3J5X21heCc6IDAuMCwKICAgICAgICAgICAgICAnd29yc3RfY29tcG9zaXRpb25fY2FzZSc6IE5vbmV9CiAgICBmb3IgaCwgdyBpbiAoKDEsIDEpLCAoNjMsIDk3KSwgKDMyMCwgMzIwKSk6CiAgICAgICAgY29sb3IgPSBybmcucmFuZG9tKChoLCB3LCAzKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICBoaXN0b3J5ID0gcm5nLnJhbmRvbSgoaCwgdywgMyksIGR0eXBlPW5wLmZsb2F0MzIpCiAgICAgICAgbW90aW9uID0gcm5nLnVuaWZvcm0oLTAuMDMsIDAuMDMsIChoLCB3LCAyKSkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgY29uZmlkZW5jZSA9IHJuZy5yYW5kb20oKGgsIHcsIDEpLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIGNvbmZpZGVuY2UucmVzaGFwZSgtMSlbOjozXSA9IDAKICAgICAgICBjb25maWRlbmNlLnJlc2hhcGUoLTEpWzE6OjNdID0gMQogICAgICAgIGdlb21ldHJ5ID0gZi5OZXR3b3JrR2VvbWV0cnkudmVuZG9yX2FsaWduZWQodywgaCkKICAgICAgICBidWlsZGVyID0gRmVhdHVyZUJ1aWxkZXIoZ2VvbWV0cnksIGRldmljZSkKICAgICAgICBjb250cm9scyA9IGYuUFJPRklMRVNbJ25hdHVyYWwnXQogICAgICAgIGZvciBpbmRleCBpbiAoMCwgMSwgMTcpOgogICAgICAgICAgICB0ZW1wb3JhbCA9IGluZGV4ICE9IDAKICAgICAgICAgICAga3dhcmdzID0ge30gaWYgbm90IHRlbXBvcmFsIGVsc2UgZGljdChoaXN0b3J5PXRvcmNoLmFzX3RlbnNvcihoaXN0b3J5LCBkZXZpY2U9ZGV2aWNlKSwKICAgICAgICAgICAgICAgIG1vdGlvbj10b3JjaC5hc190ZW5zb3IobW90aW9uLCBkZXZpY2U9ZGV2aWNlKSwgY29uZmlkZW5jZT10b3JjaC5hc190ZW5zb3IoY29uZmlkZW5jZSwgZGV2aWNlPWRldmljZSkpCiAgICAgICAgICAgIGFjdHVhbCwgaGlzdG9yeV9mID0gYnVpbGRlci5idWlsZCh0b3JjaC5hc190ZW5zb3IoY29sb3IsIGRldmljZT1kZXZpY2UpLCBpbmRleCwgY29udHJvbHMsICoqa3dhcmdzKQogICAgICAgICAgICBpZiB0ZW1wb3JhbDoKICAgICAgICAgICAgICAgIGxvZ2ljYWwgPSB0Lm1ha2VfdGVtcG9yYWxfZmVhdHVyZXMoY29sb3IsIGhpc3RvcnksIG1vdGlvbiwgZnJhbWVfaW5kZXg9aW5kZXgsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGhpc3RvcnlfY29uZmlkZW5jZT1jb25maWRlbmNlLCAqKmNvbnRyb2xzKQogICAgICAgICAgICAgICAgZXhwZWN0ZWQgPSB0LmV4dGVuZF9mZWF0dXJlcyhsb2dpY2FsLCBnZW9tZXRyeSwgaW5kZXgpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBleHBlY3RlZCA9IGYubWFrZV9mZWF0dXJlcyhjb2xvciwgZnJhbWVfaW5kZXg9aW5kZXgsIGdlb21ldHJ5PWdlb21ldHJ5LCAqKmNvbnRyb2xzKQogICAgICAgICAgICAgICAgbG9naWNhbCA9IGV4cGVjdGVkWzpoLCA6d10KICAgICAgICAgICAgYWN0dWFsX25wID0gYWN0dWFsWzBdLmZsb2F0KCkuY3B1KCkubnVtcHkoKQogICAgICAgICAgICBkZWx0YSA9IG5wLmFicyhhY3R1YWxfbnAgLSBleHBlY3RlZC5hc3R5cGUobnAuZmxvYXQxNikuYXN0eXBlKG5wLmZsb2F0MzIpKQogICAgICAgICAgICBpZiBub3QgbnAuaXNmaW5pdGUoYWN0dWFsX25wKS5hbGwoKToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignTm9uZmluaXRlIEdQVSBmZWF0dXJlIGNvbnN0cnVjdGlvbicpCiAgICAgICAgICAgIHJlcG9ydFsnbm9pc2VfbWF4J10gPSBtYXgocmVwb3J0Wydub2lzZV9tYXgnXSwgZmxvYXQoZGVsdGFbLi4uLCA6M10ubWF4KCkpKQogICAgICAgICAgICByZXBvcnRbJ25vaXNlX21hZSddID0gbWF4KHJlcG9ydFsnbm9pc2VfbWFlJ10sIGZsb2F0KGRlbHRhWy4uLiwgOjNdLm1lYW4oKSkpCiAgICAgICAgICAgIHJlcG9ydFsnZmVhdHVyZV9ub25ub2lzZV9tYXgnXSA9IG1heChyZXBvcnRbJ2ZlYXR1cmVfbm9ubm9pc2VfbWF4J10sIGZsb2F0KGRlbHRhWy4uLiwgMzpdLm1heCgpKSkKICAgICAgICAgICAgcmVwb3J0WydoaXN0b3J5X2ZlYXR1cmVzX21heCddID0gbWF4KHJlcG9ydFsnaGlzdG9yeV9mZWF0dXJlc19tYXgnXSwKICAgICAgICAgICAgICAgIGZsb2F0KG5wLmFicyhoaXN0b3J5X2YuY3B1KCkubnVtcHkoKSAtIGxvZ2ljYWxbLi4uLCA3OjEwXSkubWF4KCkpKQogICAgICAgICAgICBoZWFkID0gcm5nLm5vcm1hbCgwLCAwLjIsIChoLCB3LCA0KSkuYXN0eXBlKG5wLmZsb2F0MTYpLmFzdHlwZShucC5mbG9hdDMyKQogICAgICAgICAgICBmb3IgaW50ZW5zaXR5IGluICgwLjAsIDAuOCwgMS4wKToKICAgICAgICAgICAgICAgIGFjdHVhbF9yZ2IgPSBjb21wb3NlKHRvcmNoLmFzX3RlbnNvcihoZWFkLCBkZXZpY2U9ZGV2aWNlKSwgdG9yY2guYXNfdGVuc29yKGNvbG9yLCBkZXZpY2U9ZGV2aWNlKSwKICAgICAgICAgICAgICAgICAgICBoaXN0b3J5X2YsIHRvcmNoLmFzX3RlbnNvcihjb25maWRlbmNlLCBkZXZpY2U9ZGV2aWNlKSwgdGVtcG9yYWw9dGVtcG9yYWwsCiAgICAgICAgICAgICAgICAgICAgaW50ZW5zaXR5PWludGVuc2l0eSwgYmxlbmRfc2NhbGU9dC5CTEVORF9TQ0FMRSkuY3B1KCkubnVtcHkoKQogICAgICAgICAgICAgICAgZXhwZWN0ZWRfcmdiID0gKHQuY29tcG9zZV90ZW1wb3JhbChoZWFkLCBjb2xvciwgbG9naWNhbCwgaGlzdG9yeV9jb25maWRlbmNlPWNvbmZpZGVuY2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGludGVuc2l0eT1pbnRlbnNpdHkpIGlmIHRlbXBvcmFsIGVsc2UgY29tcG9zZV9oZWFkKGhlYWQsIGNvbG9yLCBpbnRlbnNpdHk9aW50ZW5zaXR5KSkKICAgICAgICAgICAgICAgIGVycm9yID0gZmxvYXQobnAuYWJzKGFjdHVhbF9yZ2ItZXhwZWN0ZWRfcmdiKS5tYXgoKSkKICAgICAgICAgICAgICAgIGlmIGVycm9yID4gcmVwb3J0Wydjb21wb3NpdGlvbl9tYXgnXToKICAgICAgICAgICAgICAgICAgICByZXBvcnRbJ2NvbXBvc2l0aW9uX21heCddID0gZXJyb3IKICAgICAgICAgICAgICAgICAgICByZXBvcnRbJ3dvcnN0X2NvbXBvc2l0aW9uX2Nhc2UnXSA9IHsnaGVpZ2h0JzogaCwgJ3dpZHRoJzogdywgJ2ZyYW1lX2luZGV4JzogaW5kZXgsICdpbnRlbnNpdHknOiBpbnRlbnNpdHl9CiAgICAgICAgICAgICAgICBzYW1lX2hpc3RvcnlfcmdiID0gY29tcG9zZSh0b3JjaC5hc190ZW5zb3IoaGVhZCwgZGV2aWNlPWRldmljZSksIHRvcmNoLmFzX3RlbnNvcihjb2xvciwgZGV2aWNlPWRldmljZSksCiAgICAgICAgICAgICAgICAgICAgdG9yY2guYXNfdGVuc29yKG5wLmFzY29udGlndW91c2FycmF5KGxvZ2ljYWxbLi4uLCA3OjEwXSksIGRldmljZT1kZXZpY2UpLAogICAgICAgICAgICAgICAgICAgIHRvcmNoLmFzX3RlbnNvcihjb25maWRlbmNlLCBkZXZpY2U9ZGV2aWNlKSwgdGVtcG9yYWw9dGVtcG9yYWwsCiAgICAgICAgICAgICAgICAgICAgaW50ZW5zaXR5PWludGVuc2l0eSwgYmxlbmRfc2NhbGU9dC5CTEVORF9TQ0FMRSkuY3B1KCkubnVtcHkoKQogICAgICAgICAgICAgICAgcmVwb3J0Wydjb21wb3NpdGlvbl9zYW1lX2hpc3RvcnlfbWF4J10gPSBtYXgocmVwb3J0Wydjb21wb3NpdGlvbl9zYW1lX2hpc3RvcnlfbWF4J10sCiAgICAgICAgICAgICAgICAgICAgZmxvYXQobnAuYWJzKHNhbWVfaGlzdG9yeV9yZ2IgLSBleHBlY3RlZF9yZ2IpLm1heCgpKSkKICAgIGZvciBmYWN0b3IgaW4gKDEsIDIsIDMsIDQpOgogICAgICAgIGltYWdlID0gcm5nLnJhbmRvbSgoMTYqZmFjdG9yLCAyNCpmYWN0b3IsIDMpLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIGFjdHVhbCA9IGludGVnZXJfZG93bnNhbXBsZSh0b3JjaC5hc190ZW5zb3IoaW1hZ2UsIGRldmljZT1kZXZpY2UpLCAxNiwgMjQpLmNwdSgpLm51bXB5KCkKICAgICAgICByZXBvcnRbJ3Jlc2l6ZV9tYXgnXSA9IG1heChyZXBvcnRbJ3Jlc2l6ZV9tYXgnXSwgZmxvYXQobnAuYWJzKGFjdHVhbC1yZXNhbXBsZShpbWFnZSwyNCwxNikpLm1heCgpKSkKICAgICMgRmxvYXQzMiB0cmFuc2NlbmRlbnRhbCBsaWJyYXJpZXMgY2FuIGRpZmZlciBhdCBoYWxmLXF1YW50aXphdGlvbiBlZGdlcy4KICAgICMgVG9sZXJhbmNlcyBhcmUgbmFycm93OyBhIHJlYWwtZnJhbWUgY29tcGFyaXNvbiBhZGRpdGlvbmFsbHkgZ2F0ZXMgZXhwb3J0LgogICAgaWYgKHJlcG9ydFsnbm9pc2VfbWF4J10gPiAwLjAwOCBvciByZXBvcnRbJ25vaXNlX21hZSddID4gMmUtNSBvcgogICAgICAgIHJlcG9ydFsnZmVhdHVyZV9ub25ub2lzZV9tYXgnXSA+IDAuMDAwMjUgb3IgcmVwb3J0Wydjb21wb3NpdGlvbl9tYXgnXSA+IDJlLTUgb3IgcmVwb3J0WydyZXNpemVfbWF4J10gPiAyZS02KToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZidSZXNpZGVudCBmZWF0dXJlL2NvbXBvc2l0aW9uIHBhcml0eSBmYWlsZWQ6IHtyZXBvcnR9JykKICAgIHJlcG9ydFsncGFzc2VkJ10gPSBUcnVlCiAgICByZXR1cm4gcmVwb3J0Cg=='))
(FAST_RUNTIME_DIR / 'fast_pipeline.py').write_bytes(base64.b64decode('...'))

if str(FAST_RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(FAST_RUNTIME_DIR))
# Reload updated embedded sources when this cell is rerun.
for name in ('fast_pipeline', 'fast_kernels', 'graph_replay', 'fast_temporal', 'resident_temporal'):
    sys.modules.pop(name, None)
print('Fast runner ready. Continue to the weights cell, then benchmark.')


In [ ]:
# Cell 3 — User-supplied logical weights or compatible DLL.
import hashlib
from safetensors import safe_open
from mlxdlss.tools.extract_dlssnr_weights import extract_pe_resource
if not ACCEPT_MODEL_TERMS:
    raise RuntimeError('Review your rights to the NVIDIA-derived weights, then set ACCEPT_MODEL_TERMS=True in Cell 1.')
def sha256_file(path):
    with path.open('rb') as source:
        return hashlib.file_digest(source, 'sha256').hexdigest()

if not WEIGHTS_PATH.is_file():
    if not DLSSNR_DLL_PATH.is_file():
        raise FileNotFoundError('Supply your own logical weights or compatible nvngx_dlssnr.dll and set its path in Cell 1.')
    dll_sha = sha256_file(DLSSNR_DLL_PATH)
    resource_sha = hashlib.sha256(extract_pe_resource(DLSSNR_DLL_PATH.read_bytes())).hexdigest()
    expected_resource_sha = '836f445d06ecd2e59bb9f17b84b91c143396fd76ccda1c9dc7fe81d5edd548f4'
    print('DLL SHA-256:', dll_sha, '| model resource SHA-256:', resource_sha)
    if resource_sha != expected_resource_sha:
        raise RuntimeError('DLL lacks the exact model resource expected by this pinned implementation.')
    WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    packed = WEIGHTS_PATH.parent / 'dlssnr-weights-packed.safetensors'
    if not packed.is_file():
        subprocess.run(['mlxdlss-weights', 'extract', str(DLSSNR_DLL_PATH), str(packed)], check=True)
    else:
        with safe_open(str(packed), framework='numpy') as existing:
            if (existing.metadata() or {}).get('resource_sha256') != expected_resource_sha:
                raise RuntimeError('Cached packed weights came from a different model; move them aside before rerunning.')
    subprocess.run(['mlxdlss-weights', 'decode', str(packed), str(WEIGHTS_PATH)], check=True)
with safe_open(str(WEIGHTS_PATH), framework='pt', device='cpu') as weights:
    keys = list(weights.keys())
if not keys:
    raise RuntimeError('Weight file is empty.')
print('Weight file:', WEIGHTS_PATH, '| tensors:', len(keys), '| MiB:', round(WEIGHTS_PATH.stat().st_size / 1048576))


In [ ]:
# Cell 4 — Required kernel tests, real-frame comparison, and speed benchmark.
from fast_pipeline import FastEngine
ENHANCE_OPTIONS = dict(profile=PROFILE, processing_scale=PROCESSING_SCALE,
                       intensity=INTENSITY, detail_strength=DETAIL_STRENGTH,
                       colour_strength=COLOUR_STRENGTH)
# Release an earlier engine if you rerun this cell; preview/full reuse the new one.
if 'engine' in globals():
    del engine
    import gc
    gc.collect()
    torch.cuda.empty_cache()
engine = FastEngine(WEIGHTS_PATH, device=RESOLVED_DEVICE,
                    enhance=ENHANCE_OPTIONS, temporal=TEMPORAL, motion_max_side=MOTION_MAX_SIDE,
                    resident_temporal=GPU_RESIDENT_TEMPORAL)
BENCHMARK_REPORT = OUTPUT_DIR / f'{INPUT_PATH.stem}_{RUN_ID}_benchmark.json'
benchmark = engine.benchmark(INPUT_PATH, BENCHMARK_REPORT,
                             count=BENCHMARK_FRAMES, target_seconds=NATIVE_TARGET_SECONDS)
print('Measured portable speedup:', round(benchmark['measured_sample_speedup'], 2), 'x')
estimate = benchmark['estimated_processing_seconds_without_video_io']
if estimate is not None:
    print(f'Estimated full-clip compute: {estimate:.1f}s. Video I/O adds time; full export is the definitive test.')
print('Report:', BENCHMARK_REPORT)
if PROFILE_NETWORK:
    import time
    _profile_started = time.perf_counter()
    try:
        engine.profile(OUTPUT_DIR / f'{RUN_ID}_profile')
    finally:
        engine.calibration_seconds += time.perf_counter() - _profile_started
if benchmark['shape'][0] < 720 and benchmark['shape'][1] < 720:
    print('LOW-RES INPUT: this is not a 720p speed comparison. Use the actual original 720p clip, not an upscaled test.')


In [ ]:
# Cell 5 — Preview using the same loaded, validated model.
from fast_pipeline import IMAGES
from IPython.display import display, Video
from PIL import Image
IS_IMAGE = INPUT_PATH.suffix.lower() in IMAGES
PREVIEW_PATH = OUTPUT_DIR / f'{INPUT_PATH.stem}_{RUN_ID}_fast_preview{ ".png" if IS_IMAGE else ".mp4" }'
preview_metrics = engine.process(INPUT_PATH, PREVIEW_PATH,
                                 limit=None if IS_IMAGE else PREVIEW_FRAMES,
                                 target_seconds=NATIVE_TARGET_SECONDS, crf=CRF)
display(Image.open(PREVIEW_PATH) if IS_IMAGE else Video(str(PREVIEW_PATH), embed=True))


In [ ]:
# Cell 6 — Complete export and actual speed verdict.
APPROVE_PREVIEW = False  # Inspect Cell 5, then set True.
if IS_IMAGE:
    print('Image completed:', PREVIEW_PATH)
else:
    if not APPROVE_PREVIEW:
        raise RuntimeError('Inspect Cell 5, then set APPROVE_PREVIEW=True.')
    FINAL_PATH = OUTPUT_DIR / f'{INPUT_PATH.stem}_{RUN_ID}_portable_fast.mp4'
    full_metrics = engine.process(INPUT_PATH, FINAL_PATH,
                                   target_seconds=NATIVE_TARGET_SECONDS, crf=CRF)
    print('Full video:', FINAL_PATH)
    print('Measured timings:', FINAL_PATH.with_suffix('.timings.json'))
